# Exploratory anatomy-constrained classifier training

**Public role:** Public exploratory model-development notebook; not part of the required article reproduction path.

**Relation to the manuscript:** Retained for transparency. These HYGD-based exploratory models were not used to define, select, threshold, or calibrate the frozen classifier panel reported in the final manuscript.

**Protocol boundary:** This notebook is an exploratory development branch. It is published for research transparency but is excluded from the final article reproduction sequence and must not be interpreted as HYGD-driven selection of the frozen article classifier panel.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `10_anatomy_constrained_classifier_training.ipynb`


In [ ]:
# Cell 1: Setup for anatomy-constrained classifier training

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import platform
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass


def parse_bool_series(series):
    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "y": True,
                "valid": True,
                "false": False,
                "0": False,
                "no": False,
                "n": False,
                "nan": False,
                "none": False,
                "": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


def safe_file_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False
    except Exception:
        return False


def non_empty_path_series(series):
    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return ~normalized.isin(["", "nan", "none", "null"])


PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

PROJECT_DATA_DIR = PROJECT_ROOT / "project_data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_06_DIR = RESULTS_DIR / "06_hygd_final_od_oc_proxy"
RESULTS_07_DIR = RESULTS_DIR / "07_hygd_glaucoma_classifier_benchmark"
RESULTS_08_DIR = RESULTS_DIR / "08_anatomical_explainability_validation"
RESULTS_09_DIR = RESULTS_DIR / "09_anatomy_ablation_and_shortcut_controls"
RESULTS_10_DIR = RESULTS_DIR / "10_anatomy_constrained_classifier_training"

RESULTS_10_DIR.mkdir(parents=True, exist_ok=True)

ANATOMY_CONSTRAINED_DATA_DIR = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
)

ANATOMY_CONSTRAINED_DATA_DIR.mkdir(parents=True, exist_ok=True)

ANATOMY_CONSTRAINED_MODELS_DIR = (
    MODELS_DIR
    / "hygd_anatomy_constrained_classifiers"
)

ANATOMY_CONSTRAINED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK06_FINAL_STATUS_JSON = RESULTS_06_DIR / "notebook06_final_status.json"
NOTEBOOK07_FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"
NOTEBOOK08_FINAL_STATUS_JSON = RESULTS_08_DIR / "notebook08_final_status.json"
NOTEBOOK09_FINAL_STATUS_JSON = RESULTS_09_DIR / "notebook09_final_status.json"

ANALYSIS_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_final_od_oc_proxy"
    / "hygd_final_od_oc_proxy_analysis_manifest.csv"
)

SPLIT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_glaucoma_classifier_splits"
    / "hygd_image_level_split_manifest_seed42.csv"
)

FULL_IMAGE_FINALIST_REGISTRY_CSV = (
    RESULTS_08_DIR
    / "notebook08_multifinalist_xai_model_registry.csv"
)

NOTEBOOK09_FINAL_SYNTHESIS_CSV = (
    RESULTS_09_DIR
    / "notebook09_final_ablation_shortcut_synthesis.csv"
)

NOTEBOOK09_FINAL_INTERPRETATION_CSV = (
    RESULTS_09_DIR
    / "notebook09_final_interpretation_matrix.csv"
)

SETUP_SUMMARY_JSON = RESULTS_10_DIR / "notebook10_setup_summary.json"
INPUT_AUDIT_CSV = RESULTS_10_DIR / "notebook10_input_artifact_audit.csv"
EXPERIMENT_PLAN_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_experiment_plan.csv"
EXPERIMENT_PLAN_JSON = RESULTS_10_DIR / "notebook10_anatomy_constrained_experiment_plan.json"

required_inputs = [
    NOTEBOOK06_FINAL_STATUS_JSON,
    NOTEBOOK07_FINAL_STATUS_JSON,
    NOTEBOOK08_FINAL_STATUS_JSON,
    NOTEBOOK09_FINAL_STATUS_JSON,
    ANALYSIS_MANIFEST_CSV,
    SPLIT_MANIFEST_CSV,
    FULL_IMAGE_FINALIST_REGISTRY_CSV,
    NOTEBOOK09_FINAL_SYNTHESIS_CSV,
    NOTEBOOK09_FINAL_INTERPRETATION_CSV,
]

input_audit = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "path": str(path),
            "exists": safe_file_exists(path),
        }
        for path in required_inputs
    ]
)

input_audit.to_csv(INPUT_AUDIT_CSV, index=False)

missing_inputs = input_audit.loc[
    ~input_audit["exists"],
    "path",
].tolist()

if missing_inputs:
    print(input_audit.to_string(index=False))
    raise FileNotFoundError(f"Missing required Notebook 10 inputs: {missing_inputs}")

with open(NOTEBOOK06_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook06_status = json.load(f)

with open(NOTEBOOK07_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook07_status = json.load(f)

with open(NOTEBOOK08_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook08_status = json.load(f)

with open(NOTEBOOK09_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook09_status = json.load(f)

analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
split_manifest = pd.read_csv(SPLIT_MANIFEST_CSV)
full_image_registry = pd.read_csv(FULL_IMAGE_FINALIST_REGISTRY_CSV)
notebook09_synthesis = pd.read_csv(NOTEBOOK09_FINAL_SYNTHESIS_CSV)
notebook09_interpretation = pd.read_csv(NOTEBOOK09_FINAL_INTERPRETATION_CSV)

if "gon_binary" not in analysis_manifest.columns:
    raise ValueError("Missing required column in analysis manifest: gon_binary")

analysis_manifest["gon_binary"] = pd.to_numeric(
    analysis_manifest["gon_binary"],
    errors="coerce",
)

if analysis_manifest["gon_binary"].isna().any():
    missing_labels = int(analysis_manifest["gon_binary"].isna().sum())
    raise RuntimeError(f"Non-numeric gon_binary labels found: {missing_labels}")

analysis_manifest["gon_binary"] = analysis_manifest["gon_binary"].astype(int)

if "gon_label" not in analysis_manifest.columns:
    if "label" in analysis_manifest.columns:
        analysis_manifest["gon_label"] = analysis_manifest["label"].astype(str)
    else:
        analysis_manifest["gon_label"] = np.where(
            analysis_manifest["gon_binary"] == 1,
            "GON+",
            "GON-",
        )
else:
    analysis_manifest["gon_label"] = analysis_manifest["gon_label"].astype(str)

if "include_strict_anatomical_analysis" not in analysis_manifest.columns:
    if "anatomical_proxy_qc_status" in analysis_manifest.columns:
        analysis_manifest["include_strict_anatomical_analysis"] = (
            analysis_manifest["anatomical_proxy_qc_status"].astype(str) == "valid_strict"
        )
    else:
        raise ValueError("Cannot define strict anatomical cohort.")

if "include_high_confidence_anatomical_analysis" not in analysis_manifest.columns:
    if "anatomical_proxy_qc_status" in analysis_manifest.columns:
        analysis_manifest["include_high_confidence_anatomical_analysis"] = (
            analysis_manifest["anatomical_proxy_qc_status"].astype(str) == "valid_strict"
        )
    else:
        raise ValueError("Cannot define high-confidence anatomical cohort.")

required_manifest_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "image_path_for_inference",
    "mask_512_path",
    "anatomical_proxy_qc_status",
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "area_cdr_proxy",
    "quality_score",
]

missing_manifest_cols = [
    col for col in required_manifest_cols
    if col not in analysis_manifest.columns
]

if missing_manifest_cols:
    raise ValueError(f"Missing analysis manifest columns: {missing_manifest_cols}")

if "glaucoma_split" not in split_manifest.columns:
    raise ValueError("Split manifest is missing glaucoma_split column.")

analysis_manifest["sample_id"] = analysis_manifest["sample_id"].astype(str)
analysis_manifest["patient_id"] = analysis_manifest["patient_id"].astype(str)

split_manifest["sample_id"] = split_manifest["sample_id"].astype(str)
split_manifest["patient_id"] = split_manifest["patient_id"].astype(str)

analysis_manifest["include_strict_anatomical_analysis"] = parse_bool_series(
    analysis_manifest["include_strict_anatomical_analysis"]
)

analysis_manifest["include_high_confidence_anatomical_analysis"] = parse_bool_series(
    analysis_manifest["include_high_confidence_anatomical_analysis"]
)

runtime_manifest = analysis_manifest.merge(
    split_manifest[
        [
            "sample_id",
            "patient_id",
            "glaucoma_split",
        ]
    ].drop_duplicates(),
    on=["sample_id", "patient_id"],
    how="left",
    validate="one_to_one",
)

missing_split = int(runtime_manifest["glaucoma_split"].isna().sum())

if missing_split > 0:
    raise RuntimeError(f"Images without split assignment: {missing_split}")

# Do not call Path.exists over all Google Drive image/mask paths here.
# Massive Drive stat calls can abort the mounted connection.
runtime_manifest["image_path_for_inference"] = runtime_manifest["image_path_for_inference"].astype(str)
runtime_manifest["mask_512_path"] = runtime_manifest["mask_512_path"].astype(str)

runtime_manifest["image_path_non_empty"] = non_empty_path_series(
    runtime_manifest["image_path_for_inference"]
)

runtime_manifest["mask_path_non_empty"] = non_empty_path_series(
    runtime_manifest["mask_512_path"]
)

runtime_manifest["image_exists"] = runtime_manifest["image_path_non_empty"]
runtime_manifest["mask_exists"] = runtime_manifest["mask_path_non_empty"]

runtime_manifest["strict_valid_proxy"] = runtime_manifest[
    "include_strict_anatomical_analysis"
].astype(bool)

runtime_manifest["high_confidence_proxy"] = runtime_manifest[
    "include_high_confidence_anatomical_analysis"
].astype(bool)

training_ready_manifest = runtime_manifest[
    runtime_manifest["image_path_non_empty"]
    & runtime_manifest["mask_path_non_empty"]
    & runtime_manifest["strict_valid_proxy"]
].copy().reset_index(drop=True)

if len(training_ready_manifest) == 0:
    raise RuntimeError("Training-ready strict anatomical manifest is empty.")

split_summary = (
    training_ready_manifest
    .groupby("glaucoma_split")
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_gon_positive_images=("gon_binary", "sum"),
        mean_quality_score=("quality_score", "mean"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
    )
    .reset_index()
)

split_summary["n_gon_negative_images"] = (
    split_summary["n_images"]
    - split_summary["n_gon_positive_images"]
)

experiment_plan = pd.DataFrame(
    [
        {
            "experiment_id": "od_centered_crop_2p5x",
            "training_role": "primary_anatomy_constrained_model",
            "input_definition": "Square crop centered on the OD proxy, using 2.5x OD-equivalent diameter context.",
            "input_size": 384,
            "crop_scale": 2.5,
            "context_policy": "peripapillary_context_preserved",
            "background_policy": "crop_resize",
            "model_family": "efficient_cnn",
            "candidate_model": "efficientnet_b0",
            "selection_metric": "patient_val_auroc_then_brier",
            "journal_role": "Primary anatomy-constrained comparator against full-image classifiers.",
        },
        {
            "experiment_id": "od_centered_crop_4p0x",
            "training_role": "context_sensitivity_model",
            "input_definition": "Larger OD-centered crop using 4.0x OD-equivalent diameter context.",
            "input_size": 384,
            "crop_scale": 4.0,
            "context_policy": "wider_peripapillary_context_preserved",
            "background_policy": "crop_resize",
            "model_family": "efficient_cnn",
            "candidate_model": "efficientnet_b0",
            "selection_metric": "patient_val_auroc_then_brier",
            "journal_role": "Sensitivity analysis for the amount of anatomical context.",
        },
        {
            "experiment_id": "od_only_context_suppressed",
            "training_role": "strict_anatomy_stress_model",
            "input_definition": "Full image where outside-OD context is neutralized during both training and evaluation.",
            "input_size": 384,
            "crop_scale": 1.0,
            "context_policy": "outside_od_context_suppressed",
            "background_policy": "outside_od_replaced_by_local_mean",
            "model_family": "efficient_cnn",
            "candidate_model": "efficientnet_b0",
            "selection_metric": "patient_val_auroc_then_brier",
            "journal_role": "Stress-test model trained under the same distribution used at evaluation.",
        },
    ]
)

experiment_plan.to_csv(EXPERIMENT_PLAN_CSV, index=False)

with open(EXPERIMENT_PLAN_JSON, "w", encoding="utf-8") as f:
    json.dump(
        experiment_plan.to_dict(orient="records"),
        f,
        indent=2,
    )

setup_summary = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "status": "PASS",
    "objective": (
        "Train and evaluate anatomy-constrained glaucoma classifiers after Notebook 09 showed "
        "that full-image classifiers retain strong outside-OD signal."
    ),
    "device": str(DEVICE),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "project_root": str(PROJECT_ROOT),
    "notebook06_status": notebook06_status.get("status", "unknown"),
    "notebook07_status": notebook07_status.get("status", "unknown"),
    "notebook08_status": notebook08_status.get("status", "unknown"),
    "notebook09_status": notebook09_status.get("status", "unknown"),
    "notebook09_main_finding": notebook09_status.get("main_finding", ""),
    "total_manifest_images": int(len(runtime_manifest)),
    "training_ready_strict_images": int(len(training_ready_manifest)),
    "training_ready_strict_patients": int(training_ready_manifest["patient_id"].nunique()),
    "split_summary": split_summary.to_dict(orient="records"),
    "planned_experiments": experiment_plan["experiment_id"].tolist(),
    "primary_experiment": "od_centered_crop_2p5x",
    "input_audit_csv": str(INPUT_AUDIT_CSV),
    "experiment_plan_csv": str(EXPERIMENT_PLAN_CSV),
    "experiment_plan_json": str(EXPERIMENT_PLAN_JSON),
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print(f"Device: {DEVICE}")
print(f"Notebook 06 status: {setup_summary['notebook06_status']}")
print(f"Notebook 07 status: {setup_summary['notebook07_status']}")
print(f"Notebook 08 status: {setup_summary['notebook08_status']}")
print(f"Notebook 09 status: {setup_summary['notebook09_status']}")
print(f"Total manifest images: {setup_summary['total_manifest_images']}")
print(f"Training-ready strict images: {setup_summary['training_ready_strict_images']}")
print(f"Training-ready strict patients: {setup_summary['training_ready_strict_patients']}")

print("\nStrict anatomical cohort split summary:")
print(split_summary.to_string(index=False))

print("\nNotebook 09 key finding:")
print(setup_summary["notebook09_main_finding"])

print("\nAnatomy-constrained experiment plan:")
print(
    experiment_plan[
        [
            "experiment_id",
            "training_role",
            "input_definition",
            "candidate_model",
            "journal_role",
        ]
    ].to_string(index=False)
)

print(f"\nInput audit: {INPUT_AUDIT_CSV}")
print(f"Experiment plan CSV: {EXPERIMENT_PLAN_CSV}")
print(f"Experiment plan JSON: {EXPERIMENT_PLAN_JSON}")
print(f"Setup summary JSON: {SETUP_SUMMARY_JSON}")

if not runtime_manifest["image_path_non_empty"].all():
    missing_image_paths = int((~runtime_manifest["image_path_non_empty"]).sum())
    raise RuntimeError(f"Empty image paths in runtime manifest: {missing_image_paths}")

if not training_ready_manifest["mask_path_non_empty"].all():
    missing_mask_paths = int((~training_ready_manifest["mask_path_non_empty"]).sum())
    raise RuntimeError(f"Empty mask paths in training-ready strict cohort: {missing_mask_paths}")

expected_splits = {"train", "val", "test"}

if not expected_splits.issubset(set(training_ready_manifest["glaucoma_split"].astype(str))):
    raise RuntimeError("Training-ready strict cohort does not contain train/val/test splits.")

print("Status: PASS")

In [ ]:
# Cell 2: Build and visually audit anatomy-constrained classifier inputs

import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ANATOMY_INPUT_SIZE = 384
LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

ANATOMY_INPUT_ROOT = ANATOMY_CONSTRAINED_DATA_DIR / "strict_anatomical_inputs_384"
ANATOMY_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

for experiment_id in experiment_plan["experiment_id"].astype(str).tolist():
    (ANATOMY_INPUT_ROOT / experiment_id).mkdir(parents=True, exist_ok=True)

ANATOMY_INPUT_MANIFEST_CSV = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "notebook10_strict_anatomy_constrained_input_manifest.csv"
)

ANATOMY_INPUT_SUMMARY_CSV = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_input_summary.csv"
)

ANATOMY_INPUT_ERROR_LOG_CSV = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_input_error_log.csv"
)

VISUAL_AUDIT_CASES_CSV = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_visual_audit_cases.csv"
)

VISUAL_AUDIT_PANEL_PNG = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_visual_audit_panel.png"
)

VISUAL_AUDIT_PANEL_PDF = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_visual_audit_panel.pdf"
)

VISUAL_AUDIT_PANEL_SVG = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_visual_audit_panel.svg"
)

INPUT_BUILD_SUMMARY_JSON = (
    RESULTS_10_DIR
    / "notebook10_anatomy_constrained_input_build_summary.json"
)

PARTIAL_INPUT_MANIFEST_CSV = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "PARTIAL_notebook10_strict_anatomy_constrained_input_manifest.csv"
)

PARTIAL_ERROR_LOG_CSV = (
    RESULTS_10_DIR
    / "PARTIAL_notebook10_anatomy_constrained_input_error_log.csv"
)

VISUAL_AUDIT_CASES_PER_LABEL = 4
PARTIAL_SAVE_EVERY = 50


def candidate_image_paths(image_path):
    image_path = str(image_path)
    candidates = []

    local_candidate = LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name
    candidates.append(str(local_candidate))
    candidates.append(image_path)

    seen = set()
    unique_candidates = []

    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)

    return unique_candidates


def read_rgb_image_robust(image_path, n_attempts=3, sleep_sec=0.25):
    last_error = ""

    for candidate in candidate_image_paths(image_path):
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

                    return image_rgb, candidate

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(f"Could not read image from candidates for {image_path}. Last error: {last_error}")


def read_mask_robust(mask_path, n_attempts=3, sleep_sec=0.25):
    last_error = ""

    for _ in range(n_attempts):
        try:
            mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

            if mask_raw is not None:
                return mask_raw

            last_error = f"cv2.imread returned None for {mask_path}"

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(sleep_sec)

    raise FileNotFoundError(f"Could not read mask: {mask_path}. Last error: {last_error}")


def parse_grayscale_od_oc_mask(gray):
    gray = np.asarray(gray)
    positive_values = sorted([int(v) for v in np.unique(gray) if int(v) > 0])

    if len(positive_values) == 0:
        od_mask = np.zeros(gray.shape, dtype=bool)
        oc_mask = np.zeros(gray.shape, dtype=bool)

    elif len(positive_values) == 1:
        od_mask = gray == positive_values[0]
        oc_mask = np.zeros(gray.shape, dtype=bool)

    else:
        oc_value = positive_values[-1]
        od_mask = gray > 0
        oc_mask = gray == oc_value

    od_mask = od_mask | oc_mask

    return od_mask.astype(bool), oc_mask.astype(bool)


def parse_od_oc_mask(mask_raw):
    if mask_raw.ndim == 3:
        mask_rgb = cv2.cvtColor(mask_raw[:, :, :3], cv2.COLOR_BGR2RGB)
        flat = mask_rgb.reshape(-1, 3)
        unique_colors = np.unique(flat, axis=0)

        non_black_colors = [
            tuple(color.tolist())
            for color in unique_colors
            if int(color.sum()) > 0
        ]

        if len(non_black_colors) >= 2:
            color_areas = []

            for color in non_black_colors:
                color_array = np.array(color, dtype=np.uint8)
                color_mask = np.all(mask_rgb == color_array, axis=-1)
                color_areas.append((color, int(color_mask.sum())))

            color_areas = sorted(color_areas, key=lambda x: x[1], reverse=True)

            od_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)
            oc_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)

            largest_area = color_areas[0][1]

            for color, area in color_areas:
                color_array = np.array(color, dtype=np.uint8)
                current_mask = np.all(mask_rgb == color_array, axis=-1)

                if area == largest_area:
                    od_mask = od_mask | current_mask
                else:
                    oc_mask = oc_mask | current_mask
                    od_mask = od_mask | current_mask

        else:
            gray = cv2.cvtColor(mask_rgb, cv2.COLOR_RGB2GRAY)
            od_mask, oc_mask = parse_grayscale_od_oc_mask(gray)

    else:
        od_mask, oc_mask = parse_grayscale_od_oc_mask(mask_raw)

    od_mask = od_mask | oc_mask
    rim_mask = od_mask & (~oc_mask)

    return {
        "od_mask": od_mask.astype(bool),
        "oc_mask": oc_mask.astype(bool),
        "rim_mask": rim_mask.astype(bool),
        "od_plus_oc_mask": od_mask.astype(bool),
    }


def resize_mask_to_shape(mask, target_shape):
    target_h, target_w = target_shape

    resized = cv2.resize(
        mask.astype(np.uint8),
        (target_w, target_h),
        interpolation=cv2.INTER_NEAREST,
    ).astype(bool)

    return resized


def compute_fov_mask(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    fov = gray > 10

    kernel = np.ones((7, 7), np.uint8)
    fov = cv2.morphologyEx(fov.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    fov = cv2.morphologyEx(fov.astype(np.uint8), cv2.MORPH_OPEN, kernel)

    return fov.astype(bool)


def mean_color_from_region(image_rgb, region_mask):
    region_mask = region_mask.astype(bool)

    if int(region_mask.sum()) == 0:
        return image_rgb.reshape(-1, 3).mean(axis=0).astype(np.float32)

    return image_rgb[region_mask].reshape(-1, 3).mean(axis=0).astype(np.float32)


def get_mask_geometry(mask):
    ys, xs = np.where(mask.astype(bool))

    if len(xs) == 0:
        raise RuntimeError("Empty anatomical mask.")

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    center_x = float(xs.mean())
    center_y = float(ys.mean())

    area = int(mask.sum())
    equivalent_diameter = float(2.0 * np.sqrt(area / np.pi))

    return {
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "center_x": center_x,
        "center_y": center_y,
        "area": area,
        "equivalent_diameter": equivalent_diameter,
    }


def crop_square_with_padding(image_rgb, center_x, center_y, crop_side, fill_color):
    h, w = image_rgb.shape[:2]

    crop_side = int(round(crop_side))
    crop_side = max(crop_side, 32)

    x0 = int(round(center_x - crop_side / 2.0))
    y0 = int(round(center_y - crop_side / 2.0))
    x1 = x0 + crop_side
    y1 = y0 + crop_side

    source_x0 = max(x0, 0)
    source_y0 = max(y0, 0)
    source_x1 = min(x1, w)
    source_y1 = min(y1, h)

    pad_left = source_x0 - x0
    pad_top = source_y0 - y0
    pad_right = x1 - source_x1
    pad_bottom = y1 - source_y1

    canvas = np.zeros((crop_side, crop_side, 3), dtype=np.float32)
    canvas[:, :] = fill_color.astype(np.float32)

    target_x0 = pad_left
    target_y0 = pad_top
    target_x1 = target_x0 + (source_x1 - source_x0)
    target_y1 = target_y0 + (source_y1 - source_y0)

    canvas[target_y0:target_y1, target_x0:target_x1] = image_rgb[
        source_y0:source_y1,
        source_x0:source_x1,
    ].astype(np.float32)

    crop_rgb = np.clip(canvas, 0, 255).astype(np.uint8)

    metadata = {
        "crop_side_original_px": int(crop_side),
        "crop_x0": int(x0),
        "crop_y0": int(y0),
        "crop_x1": int(x1),
        "crop_y1": int(y1),
        "source_x0": int(source_x0),
        "source_y0": int(source_y0),
        "source_x1": int(source_x1),
        "source_y1": int(source_y1),
        "pad_left": int(pad_left),
        "pad_top": int(pad_top),
        "pad_right": int(pad_right),
        "pad_bottom": int(pad_bottom),
        "padding_fraction": float(
            (pad_left + pad_right + pad_top + pad_bottom)
            / max(4.0 * crop_side, 1.0)
        ),
    }

    return crop_rgb, metadata


def build_od_centered_crop(image_rgb, od_mask_original_shape, crop_scale, output_size=ANATOMY_INPUT_SIZE):
    fov_mask = compute_fov_mask(image_rgb)
    fov_mean = mean_color_from_region(image_rgb, fov_mask)

    geometry = get_mask_geometry(od_mask_original_shape)

    crop_side = crop_scale * geometry["equivalent_diameter"]

    min_side = 0.18 * min(image_rgb.shape[:2])
    crop_side = max(crop_side, min_side)

    crop_rgb, crop_metadata = crop_square_with_padding(
        image_rgb=image_rgb,
        center_x=geometry["center_x"],
        center_y=geometry["center_y"],
        crop_side=crop_side,
        fill_color=fov_mean,
    )

    crop_resized = cv2.resize(
        crop_rgb,
        (output_size, output_size),
        interpolation=cv2.INTER_AREA,
    )

    metadata = {
        **geometry,
        **crop_metadata,
        "crop_scale": float(crop_scale),
        "output_size": int(output_size),
    }

    return crop_resized, metadata


def build_od_only_context_suppressed(image_rgb, od_mask_original_shape, output_size=ANATOMY_INPUT_SIZE):
    resized_rgb = cv2.resize(
        image_rgb,
        (output_size, output_size),
        interpolation=cv2.INTER_AREA,
    )

    od_mask_384 = resize_mask_to_shape(
        od_mask_original_shape,
        target_shape=(output_size, output_size),
    )

    fov_mask = compute_fov_mask(resized_rgb)

    od_mean = mean_color_from_region(resized_rgb, od_mask_384)

    output = resized_rgb.copy().astype(np.float32)

    outside_od_fov = (~od_mask_384) & fov_mask
    output[outside_od_fov] = od_mean

    non_fov = ~fov_mask
    output[non_fov] = resized_rgb[non_fov]

    output = np.clip(output, 0, 255).astype(np.uint8)

    geometry = get_mask_geometry(od_mask_384)

    metadata = {
        **geometry,
        "crop_scale": 1.0,
        "output_size": int(output_size),
        "suppressed_pixels": int(outside_od_fov.sum()),
        "preserved_od_pixels": int(od_mask_384.sum()),
    }

    return output, metadata


def make_od_oc_overlay(image_rgb, od_mask, oc_mask, alpha=0.45):
    od_mask = od_mask.astype(bool)
    oc_mask = oc_mask.astype(bool)
    rim_mask = od_mask & (~oc_mask)

    overlay = image_rgb.copy().astype(np.float32)
    color_layer = np.zeros_like(overlay)

    color_layer[rim_mask] = np.array([255, 180, 0], dtype=np.float32)
    color_layer[oc_mask] = np.array([0, 180, 255], dtype=np.float32)

    active_mask = od_mask | oc_mask

    overlay[active_mask] = (
        (1.0 - alpha) * overlay[active_mask]
        + alpha * color_layer[active_mask]
    )

    return np.clip(overlay, 0, 255).astype(np.uint8)


def save_rgb_png(image_rgb, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(output_path), image_bgr)

    if not ok:
        raise RuntimeError(f"cv2.imwrite failed for {output_path}")

    return str(output_path)


manifest_rows = []
error_rows = []

plan_records = experiment_plan.to_dict(orient="records")

for row_idx, row in tqdm(
    training_ready_manifest.iterrows(),
    total=len(training_ready_manifest),
    desc="Building anatomy-constrained inputs",
):
    sample_id = str(row["sample_id"])

    try:
        image_rgb, resolved_image_path = read_rgb_image_robust(
            row["image_path_for_inference"]
        )

        mask_raw = read_mask_robust(row["mask_512_path"])
        masks_512 = parse_od_oc_mask(mask_raw)

        od_mask_original = resize_mask_to_shape(
            masks_512["od_mask"],
            target_shape=image_rgb.shape[:2],
        )

        oc_mask_original = resize_mask_to_shape(
            masks_512["oc_mask"],
            target_shape=image_rgb.shape[:2],
        )

        od_mask_original = od_mask_original | oc_mask_original

        if int(od_mask_original.sum()) == 0:
            raise RuntimeError("Empty OD mask after resizing to original image.")

        for plan in plan_records:
            experiment_id = str(plan["experiment_id"])
            crop_scale = float(plan["crop_scale"])

            if experiment_id in ["od_centered_crop_2p5x", "od_centered_crop_4p0x"]:
                constrained_rgb, metadata = build_od_centered_crop(
                    image_rgb=image_rgb,
                    od_mask_original_shape=od_mask_original,
                    crop_scale=crop_scale,
                    output_size=ANATOMY_INPUT_SIZE,
                )

            elif experiment_id == "od_only_context_suppressed":
                constrained_rgb, metadata = build_od_only_context_suppressed(
                    image_rgb=image_rgb,
                    od_mask_original_shape=od_mask_original,
                    output_size=ANATOMY_INPUT_SIZE,
                )

            else:
                raise ValueError(f"Unknown experiment_id: {experiment_id}")

            output_path = (
                ANATOMY_INPUT_ROOT
                / experiment_id
                / str(row["glaucoma_split"])
                / f"{sample_id}_{experiment_id}_384.png"
            )

            saved_path = save_rgb_png(
                constrained_rgb,
                output_path,
            )

            manifest_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": str(row["patient_id"]),
                    "gon_binary": int(row["gon_binary"]),
                    "gon_label": str(row["gon_label"]),
                    "glaucoma_split": str(row["glaucoma_split"]),
                    "quality_score": float(row["quality_score"]),
                    "area_cdr_proxy": float(row["area_cdr_proxy"]) if pd.notna(row["area_cdr_proxy"]) else np.nan,
                    "experiment_id": experiment_id,
                    "training_role": str(plan["training_role"]),
                    "candidate_model": str(plan["candidate_model"]),
                    "input_size": int(ANATOMY_INPUT_SIZE),
                    "crop_scale": crop_scale,
                    "context_policy": str(plan["context_policy"]),
                    "background_policy": str(plan["background_policy"]),
                    "constrained_image_path": saved_path,
                    "source_image_path": str(row["image_path_for_inference"]),
                    "resolved_image_path": str(resolved_image_path),
                    "source_mask_path": str(row["mask_512_path"]),
                    "od_area_original_px": int(od_mask_original.sum()),
                    "oc_area_original_px": int(oc_mask_original.sum()),
                    **metadata,
                }
            )

    except Exception as exc:
        error_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": str(row.get("patient_id", "")),
                "glaucoma_split": str(row.get("glaucoma_split", "")),
                "error": repr(exc),
            }
        )

    if (row_idx + 1) % PARTIAL_SAVE_EVERY == 0:
        pd.DataFrame(manifest_rows).to_csv(PARTIAL_INPUT_MANIFEST_CSV, index=False)
        pd.DataFrame(error_rows).to_csv(PARTIAL_ERROR_LOG_CSV, index=False)

input_manifest = pd.DataFrame(manifest_rows)
error_log = pd.DataFrame(error_rows)

input_manifest.to_csv(ANATOMY_INPUT_MANIFEST_CSV, index=False)
error_log.to_csv(ANATOMY_INPUT_ERROR_LOG_CSV, index=False)

input_summary = (
    input_manifest
    .groupby(["experiment_id", "glaucoma_split"])
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_gon_positive_images=("gon_binary", "sum"),
        mean_quality_score=("quality_score", "mean"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
        mean_crop_side_original_px=("crop_side_original_px", "mean"),
        mean_padding_fraction=("padding_fraction", "mean"),
        mean_od_area_original_px=("od_area_original_px", "mean"),
        mean_oc_area_original_px=("oc_area_original_px", "mean"),
    )
    .reset_index()
)

input_summary["n_gon_negative_images"] = (
    input_summary["n_images"]
    - input_summary["n_gon_positive_images"]
)

input_summary.to_csv(ANATOMY_INPUT_SUMMARY_CSV, index=False)

visual_cases = []

for split_name in ["train", "val", "test"]:
    for label_value in [1, 0]:
        block = (
            training_ready_manifest[
                (training_ready_manifest["glaucoma_split"] == split_name)
                & (training_ready_manifest["gon_binary"] == label_value)
            ]
            .copy()
            .sort_values(["area_cdr_proxy", "quality_score", "sample_id"])
            .reset_index(drop=True)
        )

        if len(block) == 0:
            continue

        selected_idx = len(block) // 2
        visual_cases.append(block.iloc[selected_idx].to_dict())

visual_cases_df = pd.DataFrame(visual_cases)

if len(visual_cases_df) > 0:
    visual_cases_df["sample_id"] = visual_cases_df["sample_id"].astype(str)
    visual_cases_df.to_csv(VISUAL_AUDIT_CASES_CSV, index=False)

    panel_columns = [
        "Original + OD/OC",
        "OD crop 2.5x",
        "OD crop 4.0x",
        "OD-only context suppressed",
    ]

    n_rows = len(visual_cases_df)
    fig, axes = plt.subplots(
        n_rows,
        len(panel_columns),
        figsize=(15, max(3.2 * n_rows, 5)),
    )

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, case_row) in enumerate(visual_cases_df.iterrows()):
        sample_id = str(case_row["sample_id"])

        image_rgb, _ = read_rgb_image_robust(case_row["image_path_for_inference"])
        mask_raw = read_mask_robust(case_row["mask_512_path"])
        masks_512 = parse_od_oc_mask(mask_raw)

        image_384 = cv2.resize(
            image_rgb,
            (ANATOMY_INPUT_SIZE, ANATOMY_INPUT_SIZE),
            interpolation=cv2.INTER_AREA,
        )

        od_mask_384 = resize_mask_to_shape(
            masks_512["od_mask"],
            target_shape=(ANATOMY_INPUT_SIZE, ANATOMY_INPUT_SIZE),
        )

        oc_mask_384 = resize_mask_to_shape(
            masks_512["oc_mask"],
            target_shape=(ANATOMY_INPUT_SIZE, ANATOMY_INPUT_SIZE),
        )

        original_overlay = make_od_oc_overlay(
            image_rgb=image_384,
            od_mask=od_mask_384,
            oc_mask=oc_mask_384,
        )

        image_paths = {
            "OD crop 2.5x": (
                ANATOMY_INPUT_ROOT
                / "od_centered_crop_2p5x"
                / str(case_row["glaucoma_split"])
                / f"{sample_id}_od_centered_crop_2p5x_384.png"
            ),
            "OD crop 4.0x": (
                ANATOMY_INPUT_ROOT
                / "od_centered_crop_4p0x"
                / str(case_row["glaucoma_split"])
                / f"{sample_id}_od_centered_crop_4p0x_384.png"
            ),
            "OD-only context suppressed": (
                ANATOMY_INPUT_ROOT
                / "od_only_context_suppressed"
                / str(case_row["glaucoma_split"])
                / f"{sample_id}_od_only_context_suppressed_384.png"
            ),
        }

        display_images = [
            original_overlay,
            read_rgb_image_robust(image_paths["OD crop 2.5x"])[0],
            read_rgb_image_robust(image_paths["OD crop 4.0x"])[0],
            read_rgb_image_robust(image_paths["OD-only context suppressed"])[0],
        ]

        row_label = (
            f"{sample_id} | {case_row['glaucoma_split']} | {case_row['gon_label']} | "
            f"Q={float(case_row['quality_score']):.2f} | "
            f"CDR={float(case_row['area_cdr_proxy']):.3f}"
        )

        for col_idx, image in enumerate(display_images):
            ax = axes[row_idx, col_idx]
            ax.imshow(image)
            ax.axis("off")

            if row_idx == 0:
                ax.set_title(panel_columns[col_idx], fontsize=11)

            if col_idx == 0:
                ax.set_ylabel(row_label, fontsize=8)

    fig.suptitle(
        "Notebook 10 visual audit: anatomy-constrained classifier inputs",
        fontsize=14,
        fontweight="bold",
        y=0.995,
    )

    plt.tight_layout()
    plt.savefig(VISUAL_AUDIT_PANEL_PNG, dpi=300, bbox_inches="tight")
    plt.savefig(VISUAL_AUDIT_PANEL_PDF, bbox_inches="tight")
    plt.savefig(VISUAL_AUDIT_PANEL_SVG, bbox_inches="tight")
    plt.show()

build_summary = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "cell": "02_build_and_visual_audit_anatomy_constrained_inputs",
    "status": "PASS",
    "input_size": int(ANATOMY_INPUT_SIZE),
    "n_training_ready_strict_images": int(len(training_ready_manifest)),
    "n_expected_manifest_rows": int(len(training_ready_manifest) * len(plan_records)),
    "n_manifest_rows": int(len(input_manifest)),
    "n_error_rows": int(len(error_log)),
    "experiments": experiment_plan["experiment_id"].astype(str).tolist(),
    "input_manifest_csv": str(ANATOMY_INPUT_MANIFEST_CSV),
    "input_summary_csv": str(ANATOMY_INPUT_SUMMARY_CSV),
    "error_log_csv": str(ANATOMY_INPUT_ERROR_LOG_CSV),
    "visual_audit_cases_csv": str(VISUAL_AUDIT_CASES_CSV),
    "visual_audit_panel_png": str(VISUAL_AUDIT_PANEL_PNG),
    "visual_audit_panel_pdf": str(VISUAL_AUDIT_PANEL_PDF),
    "visual_audit_panel_svg": str(VISUAL_AUDIT_PANEL_SVG),
}

with open(INPUT_BUILD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(build_summary, f, indent=2)

print("Anatomy-constrained input summary:")
print(input_summary.to_string(index=False))

print("\nVisual audit cases:")
if len(visual_cases_df) > 0:
    print(
        visual_cases_df[
            [
                "sample_id",
                "patient_id",
                "glaucoma_split",
                "gon_label",
                "quality_score",
                "area_cdr_proxy",
            ]
        ].to_string(index=False)
    )
else:
    print("No visual cases selected.")

print(f"\nInput manifest CSV: {ANATOMY_INPUT_MANIFEST_CSV}")
print(f"Input summary CSV: {ANATOMY_INPUT_SUMMARY_CSV}")
print(f"Error log CSV: {ANATOMY_INPUT_ERROR_LOG_CSV}")
print(f"Visual audit cases CSV: {VISUAL_AUDIT_CASES_CSV}")
print(f"Visual audit panel PNG: {VISUAL_AUDIT_PANEL_PNG}")
print(f"Visual audit panel PDF: {VISUAL_AUDIT_PANEL_PDF}")
print(f"Visual audit panel SVG: {VISUAL_AUDIT_PANEL_SVG}")
print(f"Build summary JSON: {INPUT_BUILD_SUMMARY_JSON}")

if len(error_log) > 0:
    print("\nInput construction errors:")
    print(error_log.head(20).to_string(index=False))
    raise RuntimeError(f"Anatomy-constrained input construction failed for {len(error_log)} cases.")

expected_rows = len(training_ready_manifest) * len(plan_records)

if len(input_manifest) != expected_rows:
    raise RuntimeError(
        f"Unexpected anatomy-constrained manifest rows: {len(input_manifest)} != {expected_rows}"
    )

if int((input_manifest["od_area_original_px"] <= 0).sum()) > 0:
    raise RuntimeError("One or more constrained inputs have empty OD area.")

if not Path(ANATOMY_INPUT_MANIFEST_CSV).exists():
    raise RuntimeError("Anatomy-constrained input manifest was not saved.")

print("Status: PASS")

In [ ]:
# Cell 3: Prepare runtime datasets, dataloaders, and model smoke tests

import shutil
import timm
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

ANATOMY_INPUT_MANIFEST_CSV = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "notebook10_strict_anatomy_constrained_input_manifest.csv"
)

RUNTIME_INPUT_CACHE_ROOT = Path("/content/hygd_anatomy_constrained_inputs_384")
RUNTIME_INPUT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_INPUT_MANIFEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_anatomy_constrained_input_manifest.csv"
)

RUNTIME_CACHE_SUMMARY_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_input_cache_summary.csv"
)

RUNTIME_CACHE_ERROR_LOG_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_input_cache_error_log.csv"
)

DATALOADER_SMOKE_TEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_dataloader_smoke_test.csv"
)

MODEL_SMOKE_TEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_model_smoke_test.csv"
)

TRAINING_RUNTIME_CONFIG_JSON = (
    RESULTS_10_DIR
    / "notebook10_training_runtime_config.json"
)

USE_LOCAL_RUNTIME_CACHE = True
FORCE_REBUILD_RUNTIME_CACHE = False

TRAIN_BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
EVAL_BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
NUM_WORKERS = 0
PIN_MEMORY = False

CLASSIFIER_INPUT_SIZE = 384
CANDIDATE_TIMM_MODEL = "efficientnet_b0"
USE_IMAGENET_PRETRAINED = True

if not ANATOMY_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing anatomy input manifest: {ANATOMY_INPUT_MANIFEST_CSV}")

input_manifest = pd.read_csv(ANATOMY_INPUT_MANIFEST_CSV)

required_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "glaucoma_split",
    "experiment_id",
    "constrained_image_path",
    "quality_score",
    "area_cdr_proxy",
]

missing_cols = [
    col for col in required_cols
    if col not in input_manifest.columns
]

if missing_cols:
    raise ValueError(f"Missing constrained input manifest columns: {missing_cols}")

input_manifest["sample_id"] = input_manifest["sample_id"].astype(str)
input_manifest["patient_id"] = input_manifest["patient_id"].astype(str)
input_manifest["glaucoma_split"] = input_manifest["glaucoma_split"].astype(str)
input_manifest["experiment_id"] = input_manifest["experiment_id"].astype(str)
input_manifest["constrained_image_path"] = input_manifest["constrained_image_path"].astype(str)
input_manifest["gon_binary"] = pd.to_numeric(input_manifest["gon_binary"], errors="coerce").astype(int)

expected_experiments = experiment_plan["experiment_id"].astype(str).tolist()
available_experiments = sorted(input_manifest["experiment_id"].unique().tolist())

if sorted(expected_experiments) != available_experiments:
    raise RuntimeError(
        f"Experiment mismatch. Expected {expected_experiments}, found {available_experiments}"
    )


def safe_copy_to_runtime_cache(source_path, local_path):
    source_path = Path(str(source_path))
    local_path = Path(str(local_path))
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists() and not FORCE_REBUILD_RUNTIME_CACHE:
        return str(local_path), "already_cached"

    shutil.copy2(str(source_path), str(local_path))

    return str(local_path), "copied"


cache_rows = []
cache_error_rows = []

if USE_LOCAL_RUNTIME_CACHE:
    for _, row in tqdm(
        input_manifest.iterrows(),
        total=len(input_manifest),
        desc="Caching constrained inputs locally",
    ):
        source_path = Path(str(row["constrained_image_path"]))
        local_path = (
            RUNTIME_INPUT_CACHE_ROOT
            / str(row["experiment_id"])
            / str(row["glaucoma_split"])
            / source_path.name
        )

        try:
            runtime_path, cache_status = safe_copy_to_runtime_cache(
                source_path=source_path,
                local_path=local_path,
            )

            cache_rows.append(
                {
                    "sample_id": str(row["sample_id"]),
                    "experiment_id": str(row["experiment_id"]),
                    "glaucoma_split": str(row["glaucoma_split"]),
                    "source_path": str(source_path),
                    "runtime_image_path": str(runtime_path),
                    "cache_status": cache_status,
                    "runtime_exists": Path(runtime_path).exists(),
                }
            )

        except Exception as exc:
            cache_error_rows.append(
                {
                    "sample_id": str(row["sample_id"]),
                    "experiment_id": str(row["experiment_id"]),
                    "glaucoma_split": str(row["glaucoma_split"]),
                    "source_path": str(source_path),
                    "error": repr(exc),
                }
            )

    cache_df = pd.DataFrame(cache_rows)
    cache_error_df = pd.DataFrame(cache_error_rows)

    cache_df.to_csv(RUNTIME_CACHE_SUMMARY_CSV, index=False)
    cache_error_df.to_csv(RUNTIME_CACHE_ERROR_LOG_CSV, index=False)

    if len(cache_error_df) > 0:
        print(cache_error_df.head(20).to_string(index=False))
        raise RuntimeError(f"Runtime local cache failed for {len(cache_error_df)} files.")

    input_manifest = input_manifest.merge(
        cache_df[
            [
                "sample_id",
                "experiment_id",
                "glaucoma_split",
                "runtime_image_path",
                "runtime_exists",
            ]
        ],
        on=["sample_id", "experiment_id", "glaucoma_split"],
        how="left",
        validate="one_to_one",
    )

else:
    input_manifest["runtime_image_path"] = input_manifest["constrained_image_path"]
    input_manifest["runtime_exists"] = True

    cache_df = pd.DataFrame(
        [
            {
                "sample_id": str(row["sample_id"]),
                "experiment_id": str(row["experiment_id"]),
                "glaucoma_split": str(row["glaucoma_split"]),
                "source_path": str(row["constrained_image_path"]),
                "runtime_image_path": str(row["constrained_image_path"]),
                "cache_status": "drive_path_used",
                "runtime_exists": True,
            }
            for _, row in input_manifest.iterrows()
        ]
    )

    cache_df.to_csv(RUNTIME_CACHE_SUMMARY_CSV, index=False)
    pd.DataFrame([]).to_csv(RUNTIME_CACHE_ERROR_LOG_CSV, index=False)

if input_manifest["runtime_image_path"].isna().any():
    missing_runtime_paths = int(input_manifest["runtime_image_path"].isna().sum())
    raise RuntimeError(f"Missing runtime image paths after cache preparation: {missing_runtime_paths}")

if USE_LOCAL_RUNTIME_CACHE and not input_manifest["runtime_exists"].astype(bool).all():
    missing_runtime_files = int((~input_manifest["runtime_exists"].astype(bool)).sum())
    raise RuntimeError(f"Missing local runtime cached files: {missing_runtime_files}")

input_manifest.to_csv(RUNTIME_INPUT_MANIFEST_CSV, index=False)


train_transform = transforms.Compose(
    [
        transforms.Resize((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.05,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class AnatomyConstrainedDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = str(row["runtime_image_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        label = torch.tensor(float(row["gon_binary"]), dtype=torch.float32)

        metadata = {
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "gon_label": str(row["gon_label"]),
            "glaucoma_split": str(row["glaucoma_split"]),
            "experiment_id": str(row["experiment_id"]),
            "quality_score": float(row["quality_score"]) if pd.notna(row["quality_score"]) else np.nan,
            "area_cdr_proxy": float(row["area_cdr_proxy"]) if pd.notna(row["area_cdr_proxy"]) else np.nan,
            "runtime_image_path": image_path,
        }

        return image_tensor, label, metadata


def collate_anatomy_batch(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    labels = torch.stack([item[1] for item in batch], dim=0)

    metadata = {}

    for key in batch[0][2].keys():
        metadata[key] = [item[2][key] for item in batch]

    return images, labels, metadata


def build_weighted_sampler(dataframe):
    labels = dataframe["gon_binary"].astype(int).values

    class_counts = {
        class_value: int((labels == class_value).sum())
        for class_value in [0, 1]
    }

    if class_counts[0] == 0 or class_counts[1] == 0:
        return None, class_counts

    class_weights = {
        class_value: 1.0 / class_counts[class_value]
        for class_value in [0, 1]
    }

    sample_weights = np.array(
        [
            class_weights[int(label)]
            for label in labels
        ],
        dtype=np.float64,
    )

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

    return sampler, class_counts


def get_experiment_split_df(manifest_df, experiment_id, split_name):
    split_df = manifest_df[
        (manifest_df["experiment_id"] == experiment_id)
        & (manifest_df["glaucoma_split"] == split_name)
    ].copy().reset_index(drop=True)

    if len(split_df) == 0:
        raise RuntimeError(f"Empty split dataframe: {experiment_id} | {split_name}")

    return split_df


def make_dataloaders_for_experiment(experiment_id):
    train_df = get_experiment_split_df(input_manifest, experiment_id, "train")
    val_df = get_experiment_split_df(input_manifest, experiment_id, "val")
    test_df = get_experiment_split_df(input_manifest, experiment_id, "test")

    train_dataset = AnatomyConstrainedDataset(
        dataframe=train_df,
        transform=train_transform,
    )

    val_dataset = AnatomyConstrainedDataset(
        dataframe=val_df,
        transform=eval_transform,
    )

    test_dataset = AnatomyConstrainedDataset(
        dataframe=test_df,
        transform=eval_transform,
    )

    train_sampler, train_class_counts = build_weighted_sampler(train_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_anatomy_batch,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_anatomy_batch,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_anatomy_batch,
    )

    split_dfs = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    loaders = {
        "train": train_loader,
        "val": val_loader,
        "test": test_loader,
    }

    return loaders, split_dfs, train_class_counts


def create_anatomy_classifier(
    timm_model_name=CANDIDATE_TIMM_MODEL,
    pretrained=USE_IMAGENET_PRETRAINED,
):
    model = timm.create_model(
        timm_model_name,
        pretrained=pretrained,
        num_classes=1,
    )

    return model


def flatten_logits(logits):
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    logits = logits.squeeze()

    if logits.ndim == 0:
        logits = logits.unsqueeze(0)

    return logits


def compute_sample_weighted_bce_loss(logits, labels, negative_weight, positive_weight):
    labels = labels.float()
    logits = flatten_logits(logits)

    raw_loss = F.binary_cross_entropy_with_logits(
        logits,
        labels,
        reduction="none",
    )

    weights = torch.where(
        labels >= 0.5,
        torch.tensor(float(positive_weight), device=labels.device),
        torch.tensor(float(negative_weight), device=labels.device),
    )

    weighted_loss = (raw_loss * weights).mean()

    return weighted_loss


def compute_class_loss_weights(train_df):
    n_negative = int((train_df["gon_binary"].astype(int) == 0).sum())
    n_positive = int((train_df["gon_binary"].astype(int) == 1).sum())
    n_total = n_negative + n_positive

    if n_negative == 0 or n_positive == 0:
        raise RuntimeError("Cannot compute class weights with an empty class.")

    negative_weight = n_total / (2.0 * n_negative)
    positive_weight = n_total / (2.0 * n_positive)

    return {
        "n_negative": n_negative,
        "n_positive": n_positive,
        "negative_weight": float(negative_weight),
        "positive_weight": float(positive_weight),
    }


dataloader_smoke_rows = []
model_smoke_rows = []

print(f"Device: {DEVICE}")
print(f"Runtime cache enabled: {USE_LOCAL_RUNTIME_CACHE}")
print(f"Runtime input manifest rows: {len(input_manifest)}")
print(f"Train batch size: {TRAIN_BATCH_SIZE}")
print(f"Eval batch size: {EVAL_BATCH_SIZE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"Candidate model: {CANDIDATE_TIMM_MODEL}")
print(f"ImageNet pretrained: {USE_IMAGENET_PRETRAINED}")

for experiment_id in expected_experiments:
    loaders, split_dfs, train_class_counts = make_dataloaders_for_experiment(experiment_id)

    for split_name in ["train", "val", "test"]:
        loader = loaders[split_name]
        split_df = split_dfs[split_name]

        images, labels, metadata = next(iter(loader))

        dataloader_smoke_rows.append(
            {
                "experiment_id": experiment_id,
                "split": split_name,
                "n_images": int(len(split_df)),
                "n_patients": int(split_df["patient_id"].nunique()),
                "n_gon_positive_images": int(split_df["gon_binary"].sum()),
                "n_gon_negative_images": int(len(split_df) - split_df["gon_binary"].sum()),
                "batch_image_shape": tuple(images.shape),
                "batch_label_shape": tuple(labels.shape),
                "batch_label_min": float(labels.min().item()),
                "batch_label_max": float(labels.max().item()),
                "first_sample_id": metadata["sample_id"][0],
                "first_runtime_image_path": metadata["runtime_image_path"][0],
            }
        )

    train_df = split_dfs["train"]
    class_weight_info = compute_class_loss_weights(train_df)

    model = create_anatomy_classifier().to(DEVICE)
    model.train()

    images, labels, metadata = next(iter(loaders["train"]))
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4,
    )

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        device_type="cuda",
        enabled=(DEVICE.type == "cuda"),
    ):
        logits = flatten_logits(model(images))
        loss = compute_sample_weighted_bce_loss(
            logits=logits,
            labels=labels,
            negative_weight=class_weight_info["negative_weight"],
            positive_weight=class_weight_info["positive_weight"],
        )

    loss.backward()
    optimizer.step()

    trainable_parameters = int(
        sum(p.numel() for p in model.parameters() if p.requires_grad)
    )

    model_smoke_rows.append(
        {
            "experiment_id": experiment_id,
            "candidate_model": CANDIDATE_TIMM_MODEL,
            "input_size": CLASSIFIER_INPUT_SIZE,
            "batch_shape": tuple(images.shape),
            "logits_shape": tuple(logits.shape),
            "loss": float(loss.detach().cpu().item()),
            "trainable_parameters": trainable_parameters,
            "n_train_negative": class_weight_info["n_negative"],
            "n_train_positive": class_weight_info["n_positive"],
            "negative_loss_weight": class_weight_info["negative_weight"],
            "positive_loss_weight": class_weight_info["positive_weight"],
            "status": "pass",
        }
    )

    del model, optimizer, images, labels, logits, loss

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

dataloader_smoke_df = pd.DataFrame(dataloader_smoke_rows)
model_smoke_df = pd.DataFrame(model_smoke_rows)

dataloader_smoke_df.to_csv(DATALOADER_SMOKE_TEST_CSV, index=False)
model_smoke_df.to_csv(MODEL_SMOKE_TEST_CSV, index=False)

runtime_config = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "cell": "03_prepare_runtime_datasets_dataloaders_and_model_smoke_tests",
    "status": "PASS",
    "device": str(DEVICE),
    "use_local_runtime_cache": bool(USE_LOCAL_RUNTIME_CACHE),
    "force_rebuild_runtime_cache": bool(FORCE_REBUILD_RUNTIME_CACHE),
    "runtime_input_cache_root": str(RUNTIME_INPUT_CACHE_ROOT),
    "runtime_input_manifest_csv": str(RUNTIME_INPUT_MANIFEST_CSV),
    "train_batch_size": int(TRAIN_BATCH_SIZE),
    "eval_batch_size": int(EVAL_BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "classifier_input_size": int(CLASSIFIER_INPUT_SIZE),
    "candidate_timm_model": CANDIDATE_TIMM_MODEL,
    "use_imagenet_pretrained": bool(USE_IMAGENET_PRETRAINED),
    "experiments": expected_experiments,
    "dataloader_smoke_test_csv": str(DATALOADER_SMOKE_TEST_CSV),
    "model_smoke_test_csv": str(MODEL_SMOKE_TEST_CSV),
}

with open(TRAINING_RUNTIME_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print("\nDataloader smoke test:")
print(dataloader_smoke_df.to_string(index=False))

print("\nModel smoke test:")
print(model_smoke_df.to_string(index=False))

print(f"\nRuntime input manifest: {RUNTIME_INPUT_MANIFEST_CSV}")
print(f"Runtime cache summary: {RUNTIME_CACHE_SUMMARY_CSV}")
print(f"Runtime cache error log: {RUNTIME_CACHE_ERROR_LOG_CSV}")
print(f"Dataloader smoke test CSV: {DATALOADER_SMOKE_TEST_CSV}")
print(f"Model smoke test CSV: {MODEL_SMOKE_TEST_CSV}")
print(f"Runtime config JSON: {TRAINING_RUNTIME_CONFIG_JSON}")

if len(dataloader_smoke_df) != len(expected_experiments) * 3:
    raise RuntimeError("Unexpected number of dataloader smoke-test rows.")

if len(model_smoke_df) != len(expected_experiments):
    raise RuntimeError("Unexpected number of model smoke-test rows.")

if not np.isfinite(model_smoke_df["loss"]).all():
    raise RuntimeError("Non-finite model smoke-test loss found.")

print("Status: PASS")

In [ ]:
# Cell 4: Train anatomy-constrained classifiers with resumable validation-only selection

import time
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)
from tqdm.auto import tqdm

ANATOMY_TRAINING_ROOT = RESULTS_10_DIR / "anatomy_constrained_training_runs"
ANATOMY_TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

ANATOMY_CHECKPOINT_ROOT = ANATOMY_CONSTRAINED_MODELS_DIR / "training_runs"
ANATOMY_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

TRAINING_PROGRESS_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_training_progress.csv"
VALIDATION_RANKING_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_validation_ranking.csv"
VALIDATION_SELECTED_MODEL_JSON = RESULTS_10_DIR / "notebook10_validation_selected_anatomy_constrained_model.json"
TRAINING_CONFIG_JSON = RESULTS_10_DIR / "notebook10_anatomy_constrained_training_config.json"

RUN_TRAINING_ON_CUDA_ONLY = True

MAX_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0
AMP_ENABLED = DEVICE.type == "cuda"

PRIMARY_SELECTION_METRIC = "patient_val_auroc"
SECONDARY_SELECTION_METRIC = "patient_val_brier_score"

EXPERIMENTS_TO_TRAIN = [
    "od_centered_crop_2p5x",
    "od_centered_crop_4p0x",
    "od_only_context_suppressed",
]

if DEVICE.type != "cuda" and RUN_TRAINING_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Training skipped because CUDA is not available.")
    print("Set RUN_TRAINING_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    def compute_binary_metrics_from_probs(y_true, y_prob, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)
        y_pred = (y_prob >= float(threshold)).astype(int)

        valid = np.isfinite(y_prob)
        y_true = y_true[valid]
        y_prob = y_prob[valid]
        y_pred = y_pred[valid]

        if len(y_true) == 0:
            raise RuntimeError("No valid samples for metric computation.")

        if len(np.unique(y_true)) == 2:
            auroc = float(roc_auc_score(y_true, y_prob))
            auprc = float(average_precision_score(y_true, y_prob))
            logloss = float(
                log_loss(
                    y_true,
                    np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                    labels=[0, 1],
                )
            )
        else:
            auroc = np.nan
            auprc = np.nan
            logloss = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        return {
            "auroc": auroc,
            "auprc": auprc,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "sensitivity": float(sensitivity),
            "specificity": float(specificity),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y_true, y_prob)),
            "log_loss": logloss,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


    def find_best_threshold_by_balanced_accuracy(y_true, y_prob):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)

        thresholds = np.round(np.linspace(0.01, 0.99, 99), 4)

        best_threshold = 0.5
        best_balanced_accuracy = -np.inf

        for threshold in thresholds:
            y_pred = (y_prob >= threshold).astype(int)
            score = balanced_accuracy_score(y_true, y_pred)

            if score > best_balanced_accuracy:
                best_balanced_accuracy = float(score)
                best_threshold = float(threshold)

        return best_threshold, best_balanced_accuracy


    def collect_predictions(model, loader, split_name):
        model.eval()

        prediction_rows = []

        with torch.no_grad():
            for images, labels, metadata in tqdm(
                loader,
                desc=f"Collecting {split_name} predictions",
                leave=False,
            ):
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=AMP_ENABLED,
                ):
                    logits = flatten_logits(model(images))
                    probs = torch.sigmoid(logits)

                logits_np = logits.detach().cpu().numpy()
                probs_np = probs.detach().cpu().numpy()
                labels_np = labels.detach().cpu().numpy()

                for idx in range(len(labels_np)):
                    prediction_rows.append(
                        {
                            "sample_id": str(metadata["sample_id"][idx]),
                            "patient_id": str(metadata["patient_id"][idx]),
                            "gon_label": str(metadata["gon_label"][idx]),
                            "glaucoma_split": str(metadata["glaucoma_split"][idx]),
                            "experiment_id": str(metadata["experiment_id"][idx]),
                            "quality_score": float(metadata["quality_score"][idx]),
                            "area_cdr_proxy": float(metadata["area_cdr_proxy"][idx]),
                            "runtime_image_path": str(metadata["runtime_image_path"][idx]),
                            "y_true": int(labels_np[idx]),
                            "logit": float(logits_np[idx]),
                            "y_prob": float(probs_np[idx]),
                        }
                    )

        return pd.DataFrame(prediction_rows)


    def aggregate_patient_predictions(image_predictions, aggregation_method="mean"):
        df = image_predictions.copy()

        if aggregation_method == "mean":
            patient_df = (
                df
                .groupby("patient_id")
                .agg(
                    n_images=("sample_id", "nunique"),
                    y_true=("y_true", "max"),
                    y_prob=("y_prob", "mean"),
                    y_prob_mean=("y_prob", "mean"),
                    y_prob_max=("y_prob", "max"),
                    y_prob_median=("y_prob", "median"),
                    quality_score_mean=("quality_score", "mean"),
                    area_cdr_proxy_mean=("area_cdr_proxy", "mean"),
                    experiment_id=("experiment_id", "first"),
                )
                .reset_index()
            )

        elif aggregation_method == "max":
            patient_df = (
                df
                .groupby("patient_id")
                .agg(
                    n_images=("sample_id", "nunique"),
                    y_true=("y_true", "max"),
                    y_prob=("y_prob", "max"),
                    y_prob_mean=("y_prob", "mean"),
                    y_prob_max=("y_prob", "max"),
                    y_prob_median=("y_prob", "median"),
                    quality_score_mean=("quality_score", "mean"),
                    area_cdr_proxy_mean=("area_cdr_proxy", "mean"),
                    experiment_id=("experiment_id", "first"),
                )
                .reset_index()
            )

        elif aggregation_method == "quality_weighted_mean":
            weighted_rows = []

            for patient_id, patient_block in df.groupby("patient_id"):
                weights = patient_block["quality_score"].astype(float).values
                probs = patient_block["y_prob"].astype(float).values

                if np.isfinite(weights).all() and float(weights.sum()) > 0:
                    weighted_prob = float(np.average(probs, weights=weights))
                else:
                    weighted_prob = float(np.mean(probs))

                weighted_rows.append(
                    {
                        "patient_id": str(patient_id),
                        "n_images": int(patient_block["sample_id"].nunique()),
                        "y_true": int(patient_block["y_true"].max()),
                        "y_prob": weighted_prob,
                        "y_prob_mean": float(patient_block["y_prob"].mean()),
                        "y_prob_max": float(patient_block["y_prob"].max()),
                        "y_prob_median": float(patient_block["y_prob"].median()),
                        "quality_score_mean": float(patient_block["quality_score"].mean()),
                        "area_cdr_proxy_mean": float(patient_block["area_cdr_proxy"].mean()),
                        "experiment_id": str(patient_block["experiment_id"].iloc[0]),
                    }
                )

            patient_df = pd.DataFrame(weighted_rows)

        else:
            raise ValueError(f"Unknown aggregation method: {aggregation_method}")

        patient_df["aggregation_method"] = aggregation_method

        return patient_df


    def evaluate_prediction_tables(image_predictions, patient_predictions):
        image_threshold, image_threshold_bacc = find_best_threshold_by_balanced_accuracy(
            image_predictions["y_true"].values,
            image_predictions["y_prob"].values,
        )

        patient_threshold, patient_threshold_bacc = find_best_threshold_by_balanced_accuracy(
            patient_predictions["y_true"].values,
            patient_predictions["y_prob"].values,
        )

        image_metrics = compute_binary_metrics_from_probs(
            image_predictions["y_true"].values,
            image_predictions["y_prob"].values,
            threshold=image_threshold,
        )

        patient_metrics = compute_binary_metrics_from_probs(
            patient_predictions["y_true"].values,
            patient_predictions["y_prob"].values,
            threshold=patient_threshold,
        )

        image_metrics = {
            f"image_val_{key}": value
            for key, value in image_metrics.items()
        }

        patient_metrics = {
            f"patient_val_{key}": value
            for key, value in patient_metrics.items()
        }

        combined_metrics = {
            **image_metrics,
            **patient_metrics,
            "image_val_threshold": float(image_threshold),
            "image_val_threshold_balanced_accuracy": float(image_threshold_bacc),
            "patient_val_threshold": float(patient_threshold),
            "patient_val_threshold_balanced_accuracy": float(patient_threshold_bacc),
        }

        return combined_metrics


    def should_update_best(current_metrics, best_metrics):
        if best_metrics is None:
            return True

        current_primary = float(current_metrics[PRIMARY_SELECTION_METRIC])
        best_primary = float(best_metrics[PRIMARY_SELECTION_METRIC])

        if current_primary > best_primary + 1e-8:
            return True

        if abs(current_primary - best_primary) <= 1e-8:
            current_brier = float(current_metrics[SECONDARY_SELECTION_METRIC])
            best_brier = float(best_metrics[SECONDARY_SELECTION_METRIC])

            if current_brier < best_brier - 1e-8:
                return True

            if abs(current_brier - best_brier) <= 1e-8:
                current_bacc = float(current_metrics["patient_val_balanced_accuracy"])
                best_bacc = float(best_metrics["patient_val_balanced_accuracy"])

                if current_bacc > best_bacc + 1e-8:
                    return True

        return False


    def save_checkpoint(
        checkpoint_path,
        model,
        optimizer,
        scheduler,
        epoch,
        best_epoch,
        best_metrics,
        patience_counter,
        experiment_id,
        training_history,
    ):
        checkpoint_payload = {
            "epoch": int(epoch),
            "best_epoch": int(best_epoch),
            "best_metrics": best_metrics,
            "patience_counter": int(patience_counter),
            "experiment_id": experiment_id,
            "timm_model_name": CANDIDATE_TIMM_MODEL,
            "input_size": int(CLASSIFIER_INPUT_SIZE),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "training_history": training_history,
            "seed": int(SEED),
        }

        torch.save(checkpoint_payload, checkpoint_path)


    def load_checkpoint_if_available(checkpoint_path, model, optimizer, scheduler):
        checkpoint_path = Path(checkpoint_path)

        if not checkpoint_path.exists():
            return {
                "resume": False,
                "start_epoch": 1,
                "best_epoch": 0,
                "best_metrics": None,
                "patience_counter": 0,
                "training_history": [],
            }

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        return {
            "resume": True,
            "start_epoch": int(checkpoint["epoch"]) + 1,
            "best_epoch": int(checkpoint.get("best_epoch", 0)),
            "best_metrics": checkpoint.get("best_metrics", None),
            "patience_counter": int(checkpoint.get("patience_counter", 0)),
            "training_history": checkpoint.get("training_history", []),
        }


    def train_one_experiment(experiment_id):
        experiment_output_dir = ANATOMY_TRAINING_ROOT / experiment_id
        experiment_output_dir.mkdir(parents=True, exist_ok=True)

        experiment_checkpoint_dir = ANATOMY_CHECKPOINT_ROOT / experiment_id / f"{experiment_id}_seed{SEED}"
        experiment_checkpoint_dir.mkdir(parents=True, exist_ok=True)

        best_checkpoint_path = experiment_checkpoint_dir / "best_checkpoint.pt"
        last_checkpoint_path = experiment_checkpoint_dir / "last_checkpoint.pt"

        training_log_csv = experiment_output_dir / f"notebook10_training_log_{experiment_id}.csv"
        result_csv = experiment_output_dir / f"notebook10_training_result_{experiment_id}.csv"
        val_image_predictions_csv = experiment_output_dir / f"notebook10_val_image_predictions_{experiment_id}.csv"
        val_patient_predictions_csv = experiment_output_dir / f"notebook10_val_patient_predictions_{experiment_id}.csv"
        experiment_config_json = experiment_output_dir / f"notebook10_training_config_{experiment_id}.json"

        completion_ready = (
            result_csv.exists()
            and best_checkpoint_path.exists()
            and val_image_predictions_csv.exists()
            and val_patient_predictions_csv.exists()
        )

        if completion_ready:
            result_df = pd.read_csv(result_csv)
            return {
                "experiment_id": experiment_id,
                "completed": True,
                "skipped_existing": True,
                "result_csv": str(result_csv),
                "best_checkpoint": str(best_checkpoint_path),
                "best_patient_val_auroc": float(result_df["best_patient_val_auroc"].iloc[0]),
                "best_patient_val_brier_score": float(result_df["best_patient_val_brier_score"].iloc[0]),
                "best_patient_val_balanced_accuracy": float(result_df["best_patient_val_balanced_accuracy"].iloc[0]),
                "best_epoch": int(result_df["best_epoch_one_indexed"].iloc[0]),
                "epochs_ran": int(result_df["epochs_ran"].iloc[0]),
                "elapsed_min": float(result_df["elapsed_min"].iloc[0]),
            }

        loaders, split_dfs, train_class_counts = make_dataloaders_for_experiment(experiment_id)
        class_weight_info = compute_class_loss_weights(split_dfs["train"])

        model = create_anatomy_classifier(
            timm_model_name=CANDIDATE_TIMM_MODEL,
            pretrained=USE_IMAGENET_PRETRAINED,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=MAX_EPOCHS,
        )

        try:
            scaler = torch.amp.GradScaler(
                "cuda",
                enabled=AMP_ENABLED,
            )
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(
                enabled=AMP_ENABLED,
            )

        resume_state = load_checkpoint_if_available(
            checkpoint_path=last_checkpoint_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )

        start_epoch = resume_state["start_epoch"]
        best_epoch = resume_state["best_epoch"]
        best_metrics = resume_state["best_metrics"]
        patience_counter = resume_state["patience_counter"]
        training_history = resume_state["training_history"]

        if training_log_csv.exists() and len(training_history) == 0:
            training_history = pd.read_csv(training_log_csv).to_dict(orient="records")

        experiment_config = {
            "experiment_id": experiment_id,
            "candidate_model": CANDIDATE_TIMM_MODEL,
            "input_size": int(CLASSIFIER_INPUT_SIZE),
            "use_imagenet_pretrained": bool(USE_IMAGENET_PRETRAINED),
            "max_epochs": int(MAX_EPOCHS),
            "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
            "learning_rate": float(LEARNING_RATE),
            "weight_decay": float(WEIGHT_DECAY),
            "grad_clip_norm": float(GRAD_CLIP_NORM),
            "amp_enabled": bool(AMP_ENABLED),
            "train_batch_size": int(TRAIN_BATCH_SIZE),
            "eval_batch_size": int(EVAL_BATCH_SIZE),
            "primary_selection_metric": PRIMARY_SELECTION_METRIC,
            "secondary_selection_metric": SECONDARY_SELECTION_METRIC,
            "class_weight_info": class_weight_info,
            "train_class_counts": train_class_counts,
            "best_checkpoint_path": str(best_checkpoint_path),
            "last_checkpoint_path": str(last_checkpoint_path),
            "resumed_from_checkpoint": bool(resume_state["resume"]),
            "start_epoch": int(start_epoch),
        }

        with open(experiment_config_json, "w", encoding="utf-8") as f:
            json.dump(experiment_config, f, indent=2)

        print(f"\nTraining experiment: {experiment_id}")
        print(f"Start epoch: {start_epoch}/{MAX_EPOCHS}")
        print(f"Train images: {len(split_dfs['train'])}")
        print(f"Val images: {len(split_dfs['val'])}")
        print(f"Class weights: negative={class_weight_info['negative_weight']:.6f}, positive={class_weight_info['positive_weight']:.6f}")

        start_time = time.time()

        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            model.train()

            running_loss = 0.0
            n_train_samples = 0

            train_iterator = tqdm(
                loaders["train"],
                desc=f"{experiment_id} epoch {epoch:02d}",
                leave=False,
            )

            for images, labels, metadata in train_iterator:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=AMP_ENABLED,
                ):
                    logits = flatten_logits(model(images))
                    loss = compute_sample_weighted_bce_loss(
                        logits=logits,
                        labels=labels,
                        negative_weight=class_weight_info["negative_weight"],
                        positive_weight=class_weight_info["positive_weight"],
                    )

                scaler.scale(loss).backward()

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=GRAD_CLIP_NORM,
                )

                scaler.step(optimizer)
                scaler.update()

                batch_size = int(labels.shape[0])
                running_loss += float(loss.detach().cpu().item()) * batch_size
                n_train_samples += batch_size

                train_iterator.set_postfix(
                    loss=f"{running_loss / max(n_train_samples, 1):.4f}"
                )

            scheduler.step()

            train_loss = running_loss / max(n_train_samples, 1)

            val_image_predictions = collect_predictions(
                model=model,
                loader=loaders["val"],
                split_name=f"{experiment_id} val",
            )

            val_patient_predictions = aggregate_patient_predictions(
                val_image_predictions,
                aggregation_method="mean",
            )

            val_metrics = evaluate_prediction_tables(
                image_predictions=val_image_predictions,
                patient_predictions=val_patient_predictions,
            )

            improved = should_update_best(
                current_metrics=val_metrics,
                best_metrics=best_metrics,
            )

            if improved:
                best_metrics = val_metrics.copy()
                best_epoch = int(epoch)
                patience_counter = 0

                save_checkpoint(
                    checkpoint_path=best_checkpoint_path,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    best_epoch=best_epoch,
                    best_metrics=best_metrics,
                    patience_counter=patience_counter,
                    experiment_id=experiment_id,
                    training_history=training_history,
                )

                val_image_predictions.to_csv(val_image_predictions_csv, index=False)
                val_patient_predictions.to_csv(val_patient_predictions_csv, index=False)

            else:
                patience_counter += 1

            epoch_record = {
                "experiment_id": experiment_id,
                "epoch": int(epoch),
                "train_loss": float(train_loss),
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
                "improved": bool(improved),
                "best_epoch": int(best_epoch),
                "patience_counter": int(patience_counter),
                **val_metrics,
            }

            training_history.append(epoch_record)
            pd.DataFrame(training_history).to_csv(training_log_csv, index=False)

            save_checkpoint(
                checkpoint_path=last_checkpoint_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_epoch=best_epoch,
                best_metrics=best_metrics,
                patience_counter=patience_counter,
                experiment_id=experiment_id,
                training_history=training_history,
            )

            print(
                f"{experiment_id} | epoch={epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"patient_val_auroc={val_metrics['patient_val_auroc']:.6f} | "
                f"patient_val_brier={val_metrics['patient_val_brier_score']:.6f} | "
                f"patient_val_bacc={val_metrics['patient_val_balanced_accuracy']:.6f} | "
                f"best_epoch={best_epoch} | improved={improved}"
            )

            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered for {experiment_id} at epoch {epoch}.")
                break

        elapsed_min = round((time.time() - start_time) / 60.0, 3)

        if not best_checkpoint_path.exists():
            raise RuntimeError(f"No best checkpoint saved for {experiment_id}")

        best_checkpoint = torch.load(
            best_checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        final_best_metrics = best_checkpoint["best_metrics"]

        trainable_parameters = int(
            sum(p.numel() for p in model.parameters() if p.requires_grad)
        )

        result_row = {
            "experiment_id": experiment_id,
            "candidate_model": CANDIDATE_TIMM_MODEL,
            "input_size": int(CLASSIFIER_INPUT_SIZE),
            "best_epoch_one_indexed": int(best_checkpoint["best_epoch"]),
            "epochs_ran": int(max([row["epoch"] for row in training_history])),
            "elapsed_min": float(elapsed_min),
            "trainable_parameters": trainable_parameters,
            "best_checkpoint_path": str(best_checkpoint_path),
            "last_checkpoint_path": str(last_checkpoint_path),
            "training_log_csv": str(training_log_csv),
            "val_image_predictions_csv": str(val_image_predictions_csv),
            "val_patient_predictions_csv": str(val_patient_predictions_csv),
            "best_image_val_auroc": float(final_best_metrics["image_val_auroc"]),
            "best_image_val_auprc": float(final_best_metrics["image_val_auprc"]),
            "best_image_val_balanced_accuracy": float(final_best_metrics["image_val_balanced_accuracy"]),
            "best_image_val_brier_score": float(final_best_metrics["image_val_brier_score"]),
            "best_image_val_threshold": float(final_best_metrics["image_val_threshold"]),
            "best_patient_val_auroc": float(final_best_metrics["patient_val_auroc"]),
            "best_patient_val_auprc": float(final_best_metrics["patient_val_auprc"]),
            "best_patient_val_balanced_accuracy": float(final_best_metrics["patient_val_balanced_accuracy"]),
            "best_patient_val_brier_score": float(final_best_metrics["patient_val_brier_score"]),
            "best_patient_val_threshold": float(final_best_metrics["patient_val_threshold"]),
            "best_patient_val_sensitivity": float(final_best_metrics["patient_val_sensitivity"]),
            "best_patient_val_specificity": float(final_best_metrics["patient_val_specificity"]),
        }

        pd.DataFrame([result_row]).to_csv(result_csv, index=False)

        del model, optimizer, scheduler, scaler

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        return {
            "experiment_id": experiment_id,
            "completed": True,
            "skipped_existing": False,
            "result_csv": str(result_csv),
            "best_checkpoint": str(best_checkpoint_path),
            "best_patient_val_auroc": float(result_row["best_patient_val_auroc"]),
            "best_patient_val_brier_score": float(result_row["best_patient_val_brier_score"]),
            "best_patient_val_balanced_accuracy": float(result_row["best_patient_val_balanced_accuracy"]),
            "best_epoch": int(result_row["best_epoch_one_indexed"]),
            "epochs_ran": int(result_row["epochs_ran"]),
            "elapsed_min": float(result_row["elapsed_min"]),
        }


    training_config = {
        "notebook": "10_anatomy_constrained_classifier_training",
        "cell": "04_train_anatomy_constrained_classifiers",
        "status": "configured",
        "device": str(DEVICE),
        "experiments_to_train": EXPERIMENTS_TO_TRAIN,
        "candidate_model": CANDIDATE_TIMM_MODEL,
        "input_size": int(CLASSIFIER_INPUT_SIZE),
        "use_imagenet_pretrained": bool(USE_IMAGENET_PRETRAINED),
        "max_epochs": int(MAX_EPOCHS),
        "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "grad_clip_norm": float(GRAD_CLIP_NORM),
        "amp_enabled": bool(AMP_ENABLED),
        "primary_selection_metric": PRIMARY_SELECTION_METRIC,
        "secondary_selection_metric": SECONDARY_SELECTION_METRIC,
        "training_root": str(ANATOMY_TRAINING_ROOT),
        "checkpoint_root": str(ANATOMY_CHECKPOINT_ROOT),
    }

    with open(TRAINING_CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(training_config, f, indent=2)

    progress_rows = []

    print("Anatomy-constrained training started")
    print(f"Device: {DEVICE}")
    print(f"Experiments: {EXPERIMENTS_TO_TRAIN}")
    print(f"Primary selection metric: {PRIMARY_SELECTION_METRIC}")
    print(f"Secondary selection metric: {SECONDARY_SELECTION_METRIC}")

    for experiment_id in EXPERIMENTS_TO_TRAIN:
        result = train_one_experiment(experiment_id)
        progress_rows.append(result)

        progress_df = pd.DataFrame(progress_rows)
        progress_df.to_csv(TRAINING_PROGRESS_CSV, index=False)

        print(f"\nCompleted: {experiment_id}")
        print(pd.DataFrame([result]).to_string(index=False))

    progress_df = pd.DataFrame(progress_rows)
    progress_df.to_csv(TRAINING_PROGRESS_CSV, index=False)

    result_frames = []

    for row in progress_rows:
        result_csv = Path(row["result_csv"])

        if not result_csv.exists():
            raise RuntimeError(f"Missing result CSV: {result_csv}")

        result_frames.append(pd.read_csv(result_csv))

    consolidated_results = pd.concat(result_frames, axis=0, ignore_index=True)

    consolidated_results["validation_selection_score"] = (
        0.45 * consolidated_results["best_patient_val_auroc"].astype(float)
        + 0.25 * consolidated_results["best_patient_val_auprc"].astype(float)
        + 0.20 * consolidated_results["best_patient_val_balanced_accuracy"].astype(float)
        - 0.10 * consolidated_results["best_patient_val_brier_score"].astype(float)
    )

    validation_ranking = consolidated_results.sort_values(
        [
            "validation_selection_score",
            "best_patient_val_auroc",
            "best_patient_val_brier_score",
        ],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    validation_ranking["validation_rank"] = np.arange(1, len(validation_ranking) + 1)

    validation_ranking = validation_ranking[
        [
            "validation_rank",
            "experiment_id",
            "candidate_model",
            "best_epoch_one_indexed",
            "epochs_ran",
            "elapsed_min",
            "trainable_parameters",
            "best_patient_val_auroc",
            "best_patient_val_auprc",
            "best_patient_val_balanced_accuracy",
            "best_patient_val_brier_score",
            "best_patient_val_threshold",
            "best_patient_val_sensitivity",
            "best_patient_val_specificity",
            "best_image_val_auroc",
            "best_image_val_balanced_accuracy",
            "validation_selection_score",
            "best_checkpoint_path",
            "training_log_csv",
            "val_image_predictions_csv",
            "val_patient_predictions_csv",
        ]
    ]

    validation_ranking.to_csv(VALIDATION_RANKING_CSV, index=False)

    selected_row = validation_ranking.iloc[0].to_dict()

    selected_payload = {
        "selection_stage": "validation_only_anatomy_constrained_model_selection",
        "selected_experiment_id": str(selected_row["experiment_id"]),
        "selected_candidate_model": str(selected_row["candidate_model"]),
        "selected_best_checkpoint": str(selected_row["best_checkpoint_path"]),
        "selected_best_epoch_one_indexed": int(selected_row["best_epoch_one_indexed"]),
        "selected_best_patient_val_auroc": float(selected_row["best_patient_val_auroc"]),
        "selected_best_patient_val_auprc": float(selected_row["best_patient_val_auprc"]),
        "selected_best_patient_val_balanced_accuracy": float(selected_row["best_patient_val_balanced_accuracy"]),
        "selected_best_patient_val_brier_score": float(selected_row["best_patient_val_brier_score"]),
        "selected_best_patient_val_threshold": float(selected_row["best_patient_val_threshold"]),
        "validation_selection_score": float(selected_row["validation_selection_score"]),
        "selection_score_formula": (
            "0.45*patient_AUROC + 0.25*patient_AUPRC + "
            "0.20*patient_balanced_accuracy - 0.10*patient_Brier"
        ),
        "test_set_policy": (
            "The locked test set was not used for anatomy-constrained model selection. "
            "Locked-test evaluation will be performed in the next cell."
        ),
        "validation_ranking_csv": str(VALIDATION_RANKING_CSV),
        "training_progress_csv": str(TRAINING_PROGRESS_CSV),
    }

    with open(VALIDATION_SELECTED_MODEL_JSON, "w", encoding="utf-8") as f:
        json.dump(selected_payload, f, indent=2)

    print("\nAnatomy-constrained validation ranking:")
    print(
        validation_ranking[
            [
                "validation_rank",
                "experiment_id",
                "best_epoch_one_indexed",
                "best_patient_val_auroc",
                "best_patient_val_auprc",
                "best_patient_val_balanced_accuracy",
                "best_patient_val_brier_score",
                "validation_selection_score",
            ]
        ].to_string(index=False)
    )

    print("\nValidation-selected anatomy-constrained classifier:")
    print(json.dumps(selected_payload, indent=2))

    print(f"\nTraining progress CSV: {TRAINING_PROGRESS_CSV}")
    print(f"Validation ranking CSV: {VALIDATION_RANKING_CSV}")
    print(f"Selected model JSON: {VALIDATION_SELECTED_MODEL_JSON}")
    print(f"Training config JSON: {TRAINING_CONFIG_JSON}")

    required_outputs = [
        TRAINING_PROGRESS_CSV,
        VALIDATION_RANKING_CSV,
        VALIDATION_SELECTED_MODEL_JSON,
        TRAINING_CONFIG_JSON,
    ]

    missing_outputs = [
        str(path)
        for path in required_outputs
        if not Path(path).exists()
    ]

    if missing_outputs:
        raise RuntimeError(f"Missing Cell 4 outputs: {missing_outputs}")

    if len(validation_ranking) != len(EXPERIMENTS_TO_TRAIN):
        raise RuntimeError(
            f"Unexpected validation ranking rows: {len(validation_ranking)} != {len(EXPERIMENTS_TO_TRAIN)}"
        )

    if not np.isfinite(validation_ranking["best_patient_val_auroc"]).all():
        raise RuntimeError("Non-finite validation AUROC found.")

    print("Status: PASS")

In [ ]:
# Cell 5: Locked-test evaluation for all anatomy-constrained classifiers

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)

ANATOMY_TEST_EVAL_ROOT = RESULTS_10_DIR / "anatomy_constrained_locked_test_evaluation"
ANATOMY_TEST_EVAL_ROOT.mkdir(parents=True, exist_ok=True)

VALIDATION_RANKING_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_validation_ranking.csv"
TRAINING_PROGRESS_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_training_progress.csv"
VALIDATION_SELECTED_MODEL_JSON = RESULTS_10_DIR / "notebook10_validation_selected_anatomy_constrained_model.json"

ALL_TEST_IMAGE_PREDICTIONS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_image_predictions_all_experiments.csv"
)

ALL_TEST_PATIENT_PREDICTIONS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_patient_predictions_all_experiments.csv"
)

TEST_IMAGE_METRICS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_image_level_metrics.csv"
)

TEST_PATIENT_METRICS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_patient_level_metrics.csv"
)

TEST_EVALUATION_SUMMARY_JSON = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_locked_test_evaluation_summary.json"
)

TEST_EVALUATION_PROGRESS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_locked_test_evaluation_progress.csv"
)

RUN_TEST_EVALUATION_ON_CUDA_ONLY = True
TEST_AGGREGATION_METHOD = "mean"

required_inputs = [
    VALIDATION_RANKING_CSV,
    TRAINING_PROGRESS_CSV,
    VALIDATION_SELECTED_MODEL_JSON,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 5 inputs: {missing_inputs}")

if DEVICE.type != "cuda" and RUN_TEST_EVALUATION_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Locked-test evaluation skipped because CUDA is not available.")
    print("Set RUN_TEST_EVALUATION_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    validation_ranking = pd.read_csv(VALIDATION_RANKING_CSV)
    training_progress = pd.read_csv(TRAINING_PROGRESS_CSV)

    with open(VALIDATION_SELECTED_MODEL_JSON, "r", encoding="utf-8") as f:
        validation_selected_payload = json.load(f)

    experiments_to_evaluate = validation_ranking["experiment_id"].astype(str).tolist()

    def get_experiment_result_csv(experiment_id):
        direct_path = (
            ANATOMY_TRAINING_ROOT
            / experiment_id
            / f"notebook10_training_result_{experiment_id}.csv"
        )

        if direct_path.exists():
            return direct_path

        if "result_csv" in training_progress.columns:
            block = training_progress[
                training_progress["experiment_id"].astype(str) == str(experiment_id)
            ]

            if len(block) == 1:
                progress_path = Path(str(block["result_csv"].iloc[0]))

                if progress_path.exists():
                    return progress_path

        raise FileNotFoundError(f"Could not locate training result CSV for: {experiment_id}")


    def get_validation_thresholds(ranking_row):
        experiment_id = str(ranking_row["experiment_id"])

        if "best_image_val_threshold" in ranking_row.index:
            image_threshold = float(ranking_row["best_image_val_threshold"])
        else:
            result_csv = get_experiment_result_csv(experiment_id)
            result_df = pd.read_csv(result_csv)

            if "best_image_val_threshold" not in result_df.columns:
                raise KeyError(
                    f"best_image_val_threshold not found in validation ranking or result CSV for {experiment_id}"
                )

            image_threshold = float(result_df["best_image_val_threshold"].iloc[0])

        if "best_patient_val_threshold" in ranking_row.index:
            patient_threshold = float(ranking_row["best_patient_val_threshold"])
        else:
            result_csv = get_experiment_result_csv(experiment_id)
            result_df = pd.read_csv(result_csv)

            if "best_patient_val_threshold" not in result_df.columns:
                raise KeyError(
                    f"best_patient_val_threshold not found in validation ranking or result CSV for {experiment_id}"
                )

            patient_threshold = float(result_df["best_patient_val_threshold"].iloc[0])

        return image_threshold, patient_threshold


    def compute_binary_metrics_from_probs_cell5(y_true, y_prob, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)
        threshold = float(threshold)

        valid = np.isfinite(y_prob)
        y_true = y_true[valid]
        y_prob = y_prob[valid]

        if len(y_true) == 0:
            raise RuntimeError("No valid samples for metric computation.")

        y_pred = (y_prob >= threshold).astype(int)

        if len(np.unique(y_true)) == 2:
            auroc = float(roc_auc_score(y_true, y_prob))
            auprc = float(average_precision_score(y_true, y_prob))
            logloss = float(
                log_loss(
                    y_true,
                    np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                    labels=[0, 1],
                )
            )
        else:
            auroc = np.nan
            auprc = np.nan
            logloss = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        return {
            "n_samples": int(len(y_true)),
            "n_positive": int(np.sum(y_true == 1)),
            "n_negative": int(np.sum(y_true == 0)),
            "threshold": float(threshold),
            "auroc": auroc,
            "auprc": auprc,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "sensitivity": float(sensitivity),
            "specificity": float(specificity),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y_true, y_prob)),
            "log_loss": logloss,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


    def load_best_anatomy_model(experiment_id, checkpoint_path):
        model = create_anatomy_classifier(
            timm_model_name=CANDIDATE_TIMM_MODEL,
            pretrained=False,
        ).to(DEVICE)

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        return model, checkpoint


    all_image_prediction_parts = []
    all_patient_prediction_parts = []
    image_metric_rows = []
    patient_metric_rows = []
    progress_rows = []

    print("Locked-test evaluation for anatomy-constrained classifiers")
    print(f"Device: {DEVICE}")
    print(f"Experiments: {experiments_to_evaluate}")
    print(f"Aggregation method: {TEST_AGGREGATION_METHOD}")
    print(f"Validation-selected experiment: {validation_selected_payload['selected_experiment_id']}")

    for _, ranking_row in validation_ranking.iterrows():
        experiment_id = str(ranking_row["experiment_id"])
        checkpoint_path = Path(str(ranking_row["best_checkpoint_path"]))

        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Missing checkpoint for {experiment_id}: {checkpoint_path}")

        image_threshold, patient_threshold = get_validation_thresholds(ranking_row)

        test_image_predictions_csv = (
            ANATOMY_TEST_EVAL_ROOT
            / f"notebook10_test_image_predictions_{experiment_id}.csv"
        )

        test_patient_predictions_csv = (
            ANATOMY_TEST_EVAL_ROOT
            / f"notebook10_test_patient_predictions_{experiment_id}.csv"
        )

        if test_image_predictions_csv.exists() and test_patient_predictions_csv.exists():
            test_image_predictions = pd.read_csv(test_image_predictions_csv)
            test_patient_predictions = pd.read_csv(test_patient_predictions_csv)

            skipped_existing = True

        else:
            print(f"\nEvaluating experiment on locked test: {experiment_id}")

            loaders, split_dfs, train_class_counts = make_dataloaders_for_experiment(experiment_id)
            test_loader = loaders["test"]

            model, checkpoint = load_best_anatomy_model(
                experiment_id=experiment_id,
                checkpoint_path=checkpoint_path,
            )

            test_image_predictions = collect_predictions(
                model=model,
                loader=test_loader,
                split_name=f"{experiment_id} locked test",
            )

            test_patient_predictions = aggregate_patient_predictions(
                test_image_predictions,
                aggregation_method=TEST_AGGREGATION_METHOD,
            )

            test_image_predictions.to_csv(test_image_predictions_csv, index=False)
            test_patient_predictions.to_csv(test_patient_predictions_csv, index=False)

            del model, checkpoint

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            skipped_existing = False

        test_image_predictions["experiment_id"] = experiment_id
        test_image_predictions["candidate_model"] = CANDIDATE_TIMM_MODEL
        test_image_predictions["image_threshold_from_validation"] = image_threshold
        test_image_predictions["patient_threshold_from_validation"] = patient_threshold
        test_image_predictions["y_pred_image_threshold"] = (
            test_image_predictions["y_prob"].astype(float) >= image_threshold
        ).astype(int)
        test_image_predictions["correct_image_threshold"] = (
            test_image_predictions["y_pred_image_threshold"].astype(int)
            == test_image_predictions["y_true"].astype(int)
        )

        test_patient_predictions["experiment_id"] = experiment_id
        test_patient_predictions["candidate_model"] = CANDIDATE_TIMM_MODEL
        test_patient_predictions["image_threshold_from_validation"] = image_threshold
        test_patient_predictions["patient_threshold_from_validation"] = patient_threshold
        test_patient_predictions["y_pred_patient_threshold"] = (
            test_patient_predictions["y_prob"].astype(float) >= patient_threshold
        ).astype(int)
        test_patient_predictions["correct_patient_threshold"] = (
            test_patient_predictions["y_pred_patient_threshold"].astype(int)
            == test_patient_predictions["y_true"].astype(int)
        )

        image_metrics = compute_binary_metrics_from_probs_cell5(
            y_true=test_image_predictions["y_true"].values,
            y_prob=test_image_predictions["y_prob"].values,
            threshold=image_threshold,
        )

        patient_metrics = compute_binary_metrics_from_probs_cell5(
            y_true=test_patient_predictions["y_true"].values,
            y_prob=test_patient_predictions["y_prob"].values,
            threshold=patient_threshold,
        )

        image_metric_rows.append(
            {
                "level": "image",
                "experiment_id": experiment_id,
                "candidate_model": CANDIDATE_TIMM_MODEL,
                "best_epoch_one_indexed": int(ranking_row["best_epoch_one_indexed"]),
                "validation_rank": int(ranking_row["validation_rank"]),
                "is_validation_selected": bool(
                    experiment_id == validation_selected_payload["selected_experiment_id"]
                ),
                **image_metrics,
            }
        )

        patient_metric_rows.append(
            {
                "level": "patient",
                "aggregation_method": TEST_AGGREGATION_METHOD,
                "experiment_id": experiment_id,
                "candidate_model": CANDIDATE_TIMM_MODEL,
                "best_epoch_one_indexed": int(ranking_row["best_epoch_one_indexed"]),
                "validation_rank": int(ranking_row["validation_rank"]),
                "is_validation_selected": bool(
                    experiment_id == validation_selected_payload["selected_experiment_id"]
                ),
                **patient_metrics,
            }
        )

        all_image_prediction_parts.append(test_image_predictions)
        all_patient_prediction_parts.append(test_patient_predictions)

        progress_rows.append(
            {
                "experiment_id": experiment_id,
                "candidate_model": CANDIDATE_TIMM_MODEL,
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_exists": checkpoint_path.exists(),
                "skipped_existing_predictions": bool(skipped_existing),
                "test_image_predictions_csv": str(test_image_predictions_csv),
                "test_image_predictions_exists": test_image_predictions_csv.exists(),
                "test_patient_predictions_csv": str(test_patient_predictions_csv),
                "test_patient_predictions_exists": test_patient_predictions_csv.exists(),
                "n_test_images": int(len(test_image_predictions)),
                "n_test_patients": int(test_patient_predictions["patient_id"].nunique()),
                "image_threshold_from_validation": image_threshold,
                "patient_threshold_from_validation": patient_threshold,
                "test_patient_auroc": float(patient_metrics["auroc"]),
                "test_patient_balanced_accuracy": float(patient_metrics["balanced_accuracy"]),
                "test_patient_brier_score": float(patient_metrics["brier_score"]),
            }
        )

    all_image_predictions = pd.concat(
        all_image_prediction_parts,
        axis=0,
        ignore_index=True,
    )

    all_patient_predictions = pd.concat(
        all_patient_prediction_parts,
        axis=0,
        ignore_index=True,
    )

    image_metrics_df = pd.DataFrame(image_metric_rows)
    patient_metrics_df = pd.DataFrame(patient_metric_rows)
    progress_df = pd.DataFrame(progress_rows)

    all_image_predictions.to_csv(ALL_TEST_IMAGE_PREDICTIONS_CSV, index=False)
    all_patient_predictions.to_csv(ALL_TEST_PATIENT_PREDICTIONS_CSV, index=False)
    image_metrics_df.to_csv(TEST_IMAGE_METRICS_CSV, index=False)
    patient_metrics_df.to_csv(TEST_PATIENT_METRICS_CSV, index=False)
    progress_df.to_csv(TEST_EVALUATION_PROGRESS_CSV, index=False)

    selected_test_row = patient_metrics_df[
        patient_metrics_df["is_validation_selected"].astype(bool)
    ].iloc[0].to_dict()

    primary_2p5x_test_row = patient_metrics_df[
        patient_metrics_df["experiment_id"] == "od_centered_crop_2p5x"
    ].iloc[0].to_dict()

    test_summary = {
        "notebook": "10_anatomy_constrained_classifier_training",
        "cell": "05_locked_test_evaluation_all_anatomy_constrained_classifiers",
        "status": "PASS",
        "device": str(DEVICE),
        "candidate_model": CANDIDATE_TIMM_MODEL,
        "aggregation_method": TEST_AGGREGATION_METHOD,
        "evaluated_experiments": experiments_to_evaluate,
        "validation_selected_experiment": validation_selected_payload["selected_experiment_id"],
        "validation_selected_test_patient_auroc": float(selected_test_row["auroc"]),
        "validation_selected_test_patient_auprc": float(selected_test_row["auprc"]),
        "validation_selected_test_patient_balanced_accuracy": float(selected_test_row["balanced_accuracy"]),
        "validation_selected_test_patient_brier_score": float(selected_test_row["brier_score"]),
        "primary_2p5x_test_patient_auroc": float(primary_2p5x_test_row["auroc"]),
        "primary_2p5x_test_patient_auprc": float(primary_2p5x_test_row["auprc"]),
        "primary_2p5x_test_patient_balanced_accuracy": float(primary_2p5x_test_row["balanced_accuracy"]),
        "primary_2p5x_test_patient_brier_score": float(primary_2p5x_test_row["brier_score"]),
        "test_set_policy": (
            "The locked test set is used here only after validation-only selection and training completion. "
            "All anatomy-constrained experiments are evaluated for comparison; model selection was not based on test results."
        ),
        "outputs": {
            "all_test_image_predictions_csv": str(ALL_TEST_IMAGE_PREDICTIONS_CSV),
            "all_test_patient_predictions_csv": str(ALL_TEST_PATIENT_PREDICTIONS_CSV),
            "test_image_metrics_csv": str(TEST_IMAGE_METRICS_CSV),
            "test_patient_metrics_csv": str(TEST_PATIENT_METRICS_CSV),
            "test_evaluation_progress_csv": str(TEST_EVALUATION_PROGRESS_CSV),
        },
    }

    with open(TEST_EVALUATION_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(test_summary, f, indent=2)

    print("\nAnatomy-constrained locked-test image-level metrics:")
    print(
        image_metrics_df[
            [
                "experiment_id",
                "validation_rank",
                "is_validation_selected",
                "n_samples",
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "threshold",
            ]
        ].sort_values("validation_rank").to_string(index=False)
    )

    print("\nAnatomy-constrained locked-test patient-level metrics:")
    print(
        patient_metrics_df[
            [
                "experiment_id",
                "validation_rank",
                "is_validation_selected",
                "n_samples",
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "threshold",
            ]
        ].sort_values("validation_rank").to_string(index=False)
    )

    print("\nLocked-test evaluation progress:")
    print(progress_df.to_string(index=False))

    print(f"\nAll test image predictions: {ALL_TEST_IMAGE_PREDICTIONS_CSV}")
    print(f"All test patient predictions: {ALL_TEST_PATIENT_PREDICTIONS_CSV}")
    print(f"Test image metrics: {TEST_IMAGE_METRICS_CSV}")
    print(f"Test patient metrics: {TEST_PATIENT_METRICS_CSV}")
    print(f"Evaluation progress: {TEST_EVALUATION_PROGRESS_CSV}")
    print(f"Evaluation summary JSON: {TEST_EVALUATION_SUMMARY_JSON}")

    required_outputs = [
        ALL_TEST_IMAGE_PREDICTIONS_CSV,
        ALL_TEST_PATIENT_PREDICTIONS_CSV,
        TEST_IMAGE_METRICS_CSV,
        TEST_PATIENT_METRICS_CSV,
        TEST_EVALUATION_PROGRESS_CSV,
        TEST_EVALUATION_SUMMARY_JSON,
    ]

    missing_outputs = [
        str(path)
        for path in required_outputs
        if not Path(path).exists()
    ]

    if missing_outputs:
        raise RuntimeError(f"Missing Cell 5 outputs: {missing_outputs}")

    if len(patient_metrics_df) != len(experiments_to_evaluate):
        raise RuntimeError(
            f"Unexpected patient metric rows: {len(patient_metrics_df)} != {len(experiments_to_evaluate)}"
        )

    if not np.isfinite(patient_metrics_df["auroc"]).all():
        raise RuntimeError("Non-finite patient-level AUROC found.")

    print("Status: PASS")

In [ ]:
# Cell 6: Bootstrap comparison between full-image and anatomy-constrained classifiers

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
)
from tqdm.auto import tqdm

COMPARISON_ROOT = RESULTS_10_DIR / "anatomy_vs_full_image_comparison"
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

FULL_IMAGE_PATIENT_PREDICTIONS_CSV = (
    RESULTS_09_DIR
    / "full_ablation_evaluation"
    / "notebook09_full_ablation_patient_predictions_all_models.csv"
)

ANATOMY_PATIENT_PREDICTIONS_CSV = (
    RESULTS_10_DIR
    / "anatomy_constrained_locked_test_evaluation"
    / "notebook10_anatomy_constrained_test_patient_predictions_all_experiments.csv"
)

ANATOMY_PATIENT_METRICS_CSV = (
    RESULTS_10_DIR
    / "anatomy_constrained_locked_test_evaluation"
    / "notebook10_anatomy_constrained_test_patient_level_metrics.csv"
)

COMBINED_PATIENT_PREDICTIONS_CSV = (
    COMPARISON_ROOT
    / "notebook10_combined_full_image_and_anatomy_patient_predictions.csv"
)

COMBINED_BOOTSTRAP_CI_CSV = (
    COMPARISON_ROOT
    / "notebook10_combined_patient_metric_bootstrap_ci.csv"
)

PAIRED_COMPARISON_CSV = (
    COMPARISON_ROOT
    / "notebook10_paired_anatomy_vs_full_image_comparison.csv"
)

COMPARISON_INTERPRETATION_CSV = (
    COMPARISON_ROOT
    / "notebook10_anatomy_vs_full_image_interpretation_flags.csv"
)

COMPARISON_SUMMARY_JSON = (
    COMPARISON_ROOT
    / "notebook10_anatomy_vs_full_image_comparison_summary.json"
)

PARTIAL_BOOTSTRAP_CI_CSV = (
    COMPARISON_ROOT
    / "PARTIAL_notebook10_combined_patient_metric_bootstrap_ci.csv"
)

PARTIAL_PAIRED_COMPARISON_CSV = (
    COMPARISON_ROOT
    / "PARTIAL_notebook10_paired_anatomy_vs_full_image_comparison.csv"
)

BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = SEED

COMPARISON_METRICS = [
    "auroc",
    "auprc",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
]

PRIMARY_FULL_IMAGE_REFERENCE = "full_image__EfficientNetB0"
PRIMARY_ANATOMY_REFERENCE = "anatomy__od_centered_crop_2p5x"

required_inputs = [
    FULL_IMAGE_PATIENT_PREDICTIONS_CSV,
    ANATOMY_PATIENT_PREDICTIONS_CSV,
    ANATOMY_PATIENT_METRICS_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 6 inputs: {missing_inputs}")

full_patient_predictions = pd.read_csv(FULL_IMAGE_PATIENT_PREDICTIONS_CSV)
anatomy_patient_predictions = pd.read_csv(ANATOMY_PATIENT_PREDICTIONS_CSV)
anatomy_patient_metrics = pd.read_csv(ANATOMY_PATIENT_METRICS_CSV)

full_patient_predictions["patient_id"] = full_patient_predictions["patient_id"].astype(str)
anatomy_patient_predictions["patient_id"] = anatomy_patient_predictions["patient_id"].astype(str)

full_original = full_patient_predictions[
    full_patient_predictions["ablation_mode"].astype(str) == "original_full_image"
].copy()

if len(full_original) == 0:
    raise RuntimeError("No original_full_image rows found in Notebook 09 full-image patient predictions.")

def compute_metric_bundle(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    threshold = float(threshold)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return {
            metric: np.nan
            for metric in COMPARISON_METRICS
        }

    y_pred = (y_prob >= threshold).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
    else:
        auroc = np.nan
        auprc = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    if np.isfinite(sensitivity) and np.isfinite(specificity):
        balanced_accuracy = 0.5 * (sensitivity + specificity)
    else:
        balanced_accuracy = np.nan

    return {
        "auroc": auroc,
        "auprc": auprc,
        "balanced_accuracy": float(balanced_accuracy),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
    }


def stable_seed(*parts, modulo=2**32 - 1):
    import hashlib

    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()

    return int(digest[:8], 16) % modulo


def bootstrap_ci_for_model(model_df, model_id, n_bootstrap, progress_bar=None):
    y_true = model_df["y_true"].astype(int).values
    y_prob = model_df["y_prob"].astype(float).values
    threshold = float(model_df["threshold"].iloc[0])

    observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=y_prob,
        threshold=threshold,
    )

    rng = np.random.default_rng(
        stable_seed("ci", model_id, BOOTSTRAP_SEED)
    )

    n = len(model_df)
    boot_values = {
        metric: []
        for metric in COMPARISON_METRICS
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=y_prob[idx],
            threshold=threshold,
        )

        for metric in COMPARISON_METRICS:
            value = metrics[metric]

            if np.isfinite(value):
                boot_values[metric].append(float(value))

        if progress_bar is not None:
            progress_bar.update(1)

    rows = []

    for metric in COMPARISON_METRICS:
        values = np.asarray(boot_values[metric], dtype=float)

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            n_valid = 0
        else:
            ci_low, ci_high = np.percentile(values, [2.5, 97.5])
            n_valid = len(values)

        rows.append(
            {
                "model_id": model_id,
                "model_source": str(model_df["model_source"].iloc[0]),
                "model_name": str(model_df["model_name"].iloc[0]),
                "experiment_id": str(model_df["experiment_id"].iloc[0]),
                "metric": metric,
                "observed": float(observed[metric]),
                "ci95_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
                "ci95_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
                "n_valid_bootstrap": int(n_valid),
                "n_patients": int(len(model_df)),
                "threshold": threshold,
            }
        )

    return rows


def bootstrap_paired_delta(reference_df, comparator_df, reference_id, comparator_id, n_bootstrap, progress_bar=None):
    reference_cols = [
        "patient_id",
        "y_true",
        "y_prob",
        "threshold",
    ]

    comparator_cols = [
        "patient_id",
        "y_true",
        "y_prob",
        "threshold",
    ]

    merged = reference_df[reference_cols].rename(
        columns={
            "y_prob": "reference_y_prob",
            "threshold": "reference_threshold",
        }
    ).merge(
        comparator_df[comparator_cols].rename(
            columns={
                "y_prob": "comparator_y_prob",
                "threshold": "comparator_threshold",
            }
        ),
        on=["patient_id", "y_true"],
        how="inner",
        validate="one_to_one",
    )

    if len(merged) == 0:
        raise RuntimeError(f"No paired patients for {comparator_id} vs {reference_id}")

    y_true = merged["y_true"].astype(int).values
    reference_prob = merged["reference_y_prob"].astype(float).values
    comparator_prob = merged["comparator_y_prob"].astype(float).values
    reference_threshold = float(merged["reference_threshold"].iloc[0])
    comparator_threshold = float(merged["comparator_threshold"].iloc[0])

    reference_observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=reference_prob,
        threshold=reference_threshold,
    )

    comparator_observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=comparator_prob,
        threshold=comparator_threshold,
    )

    rng = np.random.default_rng(
        stable_seed("paired", reference_id, comparator_id, BOOTSTRAP_SEED)
    )

    n = len(merged)

    boot_deltas = {
        metric: []
        for metric in COMPARISON_METRICS
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        reference_metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=reference_prob[idx],
            threshold=reference_threshold,
        )

        comparator_metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=comparator_prob[idx],
            threshold=comparator_threshold,
        )

        for metric in COMPARISON_METRICS:
            delta = comparator_metrics[metric] - reference_metrics[metric]

            if np.isfinite(delta):
                boot_deltas[metric].append(float(delta))

        if progress_bar is not None:
            progress_bar.update(1)

    rows = []

    for metric in COMPARISON_METRICS:
        values = np.asarray(boot_deltas[metric], dtype=float)

        observed_delta = comparator_observed[metric] - reference_observed[metric]

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            p_two_sided = np.nan
            n_valid = 0
        else:
            ci_low, ci_high = np.percentile(values, [2.5, 97.5])

            p_lower_or_equal_zero = np.mean(values <= 0)
            p_greater_or_equal_zero = np.mean(values >= 0)
            p_two_sided = min(
                1.0,
                2.0 * min(p_lower_or_equal_zero, p_greater_or_equal_zero),
            )

            n_valid = len(values)

        rows.append(
            {
                "reference_model_id": reference_id,
                "comparator_model_id": comparator_id,
                "metric": metric,
                "reference_observed": float(reference_observed[metric]),
                "comparator_observed": float(comparator_observed[metric]),
                "delta_comparator_minus_reference": float(observed_delta),
                "delta_ci95_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
                "delta_ci95_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
                "paired_bootstrap_p_two_sided": float(p_two_sided) if np.isfinite(p_two_sided) else np.nan,
                "n_valid_bootstrap": int(n_valid),
                "n_paired_patients": int(len(merged)),
            }
        )

    return rows


def interpret_delta(metric, delta, ci_low, ci_high):
    if metric == "brier_score":
        if delta < 0 and ci_high < 0:
            return "comparator_significantly_better"
        if delta > 0 and ci_low > 0:
            return "comparator_significantly_worse"
        return "inconclusive"

    if delta > 0 and ci_low > 0:
        return "comparator_significantly_better"
    if delta < 0 and ci_high < 0:
        return "comparator_significantly_worse"
    return "inconclusive"


combined_rows = []

for _, row in full_original.iterrows():
    model_name = str(row["model_name"])
    model_id = f"full_image__{model_name}"

    threshold_col = "patient_threshold"

    if threshold_col not in row.index:
        raise KeyError("Notebook 09 full-image patient predictions are missing patient_threshold.")

    combined_rows.append(
        {
            "model_id": model_id,
            "model_source": "full_image",
            "model_name": model_name,
            "experiment_id": "original_full_image",
            "patient_id": str(row["patient_id"]),
            "y_true": int(row["gon_binary"]),
            "y_prob": float(row["y_prob"]),
            "threshold": float(row["patient_threshold"]),
            "n_images": int(row["n_images"]),
        }
    )

for _, row in anatomy_patient_predictions.iterrows():
    experiment_id = str(row["experiment_id"])
    model_id = f"anatomy__{experiment_id}"

    combined_rows.append(
        {
            "model_id": model_id,
            "model_source": "anatomy_constrained",
            "model_name": "EfficientNetB0",
            "experiment_id": experiment_id,
            "patient_id": str(row["patient_id"]),
            "y_true": int(row["y_true"]),
            "y_prob": float(row["y_prob"]),
            "threshold": float(row["patient_threshold_from_validation"]),
            "n_images": int(row["n_images"]),
        }
    )

combined_predictions = pd.DataFrame(combined_rows)

expected_patients = sorted(
    combined_predictions[
        combined_predictions["model_id"] == PRIMARY_FULL_IMAGE_REFERENCE
    ]["patient_id"].unique().tolist()
)

if len(expected_patients) == 0:
    raise RuntimeError(f"Primary full-image reference not found: {PRIMARY_FULL_IMAGE_REFERENCE}")

model_patient_counts = (
    combined_predictions
    .groupby("model_id")
    .agg(
        n_patients=("patient_id", "nunique"),
        n_positive=("y_true", "sum"),
    )
    .reset_index()
)

if int(model_patient_counts["n_patients"].min()) != len(expected_patients):
    raise RuntimeError("Not all models contain the same number of patients.")

combined_predictions.to_csv(COMBINED_PATIENT_PREDICTIONS_CSV, index=False)

model_ids = sorted(combined_predictions["model_id"].unique().tolist())
full_model_ids = sorted(
    [model_id for model_id in model_ids if model_id.startswith("full_image__")]
)
anatomy_model_ids = sorted(
    [model_id for model_id in model_ids if model_id.startswith("anatomy__")]
)

comparison_pairs = []

for anatomy_id in anatomy_model_ids:
    for full_id in full_model_ids:
        comparison_pairs.append(
            {
                "reference_model_id": full_id,
                "comparator_model_id": anatomy_id,
            }
        )

total_steps = (
    len(model_ids) * BOOTSTRAP_N
    + len(comparison_pairs) * BOOTSTRAP_N
)

print("Bootstrap comparison: full-image vs anatomy-constrained classifiers")
print(f"Bootstrap resamples: {BOOTSTRAP_N}")
print(f"Models: {model_ids}")
print(f"Full-image references: {full_model_ids}")
print(f"Anatomy comparators: {anatomy_model_ids}")
print(f"Comparison pairs: {len(comparison_pairs)}")
print(f"Total progress steps: {total_steps}")

bootstrap_rows = []
paired_rows = []

progress_bar = tqdm(
    total=total_steps,
    desc="Anatomy vs full-image bootstrap",
)

for model_id in model_ids:
    model_df = combined_predictions[
        combined_predictions["model_id"] == model_id
    ].copy()

    bootstrap_rows.extend(
        bootstrap_ci_for_model(
            model_df=model_df,
            model_id=model_id,
            n_bootstrap=BOOTSTRAP_N,
            progress_bar=progress_bar,
        )
    )

    pd.DataFrame(bootstrap_rows).to_csv(PARTIAL_BOOTSTRAP_CI_CSV, index=False)

for pair in comparison_pairs:
    reference_id = pair["reference_model_id"]
    comparator_id = pair["comparator_model_id"]

    reference_df = combined_predictions[
        combined_predictions["model_id"] == reference_id
    ].copy()

    comparator_df = combined_predictions[
        combined_predictions["model_id"] == comparator_id
    ].copy()

    paired_rows.extend(
        bootstrap_paired_delta(
            reference_df=reference_df,
            comparator_df=comparator_df,
            reference_id=reference_id,
            comparator_id=comparator_id,
            n_bootstrap=BOOTSTRAP_N,
            progress_bar=progress_bar,
        )
    )

    pd.DataFrame(paired_rows).to_csv(PARTIAL_PAIRED_COMPARISON_CSV, index=False)

progress_bar.close()

bootstrap_ci = pd.DataFrame(bootstrap_rows)
paired_comparison = pd.DataFrame(paired_rows)

bootstrap_ci.to_csv(COMBINED_BOOTSTRAP_CI_CSV, index=False)
paired_comparison.to_csv(PAIRED_COMPARISON_CSV, index=False)

interpretation_rows = []

for _, row in paired_comparison.iterrows():
    interpretation_rows.append(
        {
            "reference_model_id": str(row["reference_model_id"]),
            "comparator_model_id": str(row["comparator_model_id"]),
            "metric": str(row["metric"]),
            "delta_comparator_minus_reference": float(row["delta_comparator_minus_reference"]),
            "delta_ci95_low": float(row["delta_ci95_low"]) if pd.notna(row["delta_ci95_low"]) else np.nan,
            "delta_ci95_high": float(row["delta_ci95_high"]) if pd.notna(row["delta_ci95_high"]) else np.nan,
            "paired_bootstrap_p_two_sided": float(row["paired_bootstrap_p_two_sided"]) if pd.notna(row["paired_bootstrap_p_two_sided"]) else np.nan,
            "interpretation": interpret_delta(
                metric=str(row["metric"]),
                delta=float(row["delta_comparator_minus_reference"]),
                ci_low=float(row["delta_ci95_low"]) if pd.notna(row["delta_ci95_low"]) else np.nan,
                ci_high=float(row["delta_ci95_high"]) if pd.notna(row["delta_ci95_high"]) else np.nan,
            ),
        }
    )

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_df.to_csv(COMPARISON_INTERPRETATION_CSV, index=False)

primary_pair_summary = paired_comparison[
    (paired_comparison["reference_model_id"] == PRIMARY_FULL_IMAGE_REFERENCE)
    & (paired_comparison["comparator_model_id"] == PRIMARY_ANATOMY_REFERENCE)
].copy()

summary_payload = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "cell": "06_bootstrap_comparison_full_image_vs_anatomy_constrained",
    "status": "PASS",
    "bootstrap_n": int(BOOTSTRAP_N),
    "n_models": int(len(model_ids)),
    "n_comparison_pairs": int(len(comparison_pairs)),
    "n_patients_per_model": int(len(expected_patients)),
    "primary_full_image_reference": PRIMARY_FULL_IMAGE_REFERENCE,
    "primary_anatomy_reference": PRIMARY_ANATOMY_REFERENCE,
    "primary_pair_metrics": primary_pair_summary.to_dict(orient="records"),
    "outputs": {
        "combined_patient_predictions_csv": str(COMBINED_PATIENT_PREDICTIONS_CSV),
        "combined_bootstrap_ci_csv": str(COMBINED_BOOTSTRAP_CI_CSV),
        "paired_comparison_csv": str(PAIRED_COMPARISON_CSV),
        "comparison_interpretation_csv": str(COMPARISON_INTERPRETATION_CSV),
        "partial_bootstrap_ci_csv": str(PARTIAL_BOOTSTRAP_CI_CSV),
        "partial_paired_comparison_csv": str(PARTIAL_PAIRED_COMPARISON_CSV),
    },
}

with open(COMPARISON_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("\nCombined patient-level bootstrap CI:")
print(
    bootstrap_ci[
        bootstrap_ci["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
    ][
        [
            "model_id",
            "metric",
            "observed",
            "ci95_low",
            "ci95_high",
            "n_valid_bootstrap",
        ]
    ].sort_values(["model_id", "metric"]).to_string(index=False)
)

print("\nPrimary paired comparison: anatomy 2.5x vs full-image EfficientNetB0")
print(
    primary_pair_summary[
        [
            "metric",
            "reference_observed",
            "comparator_observed",
            "delta_comparator_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].to_string(index=False)
)

print("\nAll paired comparisons, primary metrics:")
print(
    paired_comparison[
        paired_comparison["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
    ][
        [
            "reference_model_id",
            "comparator_model_id",
            "metric",
            "reference_observed",
            "comparator_observed",
            "delta_comparator_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].sort_values(
        [
            "reference_model_id",
            "comparator_model_id",
            "metric",
        ]
    ).to_string(index=False)
)

print(f"\nCombined predictions CSV: {COMBINED_PATIENT_PREDICTIONS_CSV}")
print(f"Bootstrap CI CSV: {COMBINED_BOOTSTRAP_CI_CSV}")
print(f"Paired comparison CSV: {PAIRED_COMPARISON_CSV}")
print(f"Interpretation CSV: {COMPARISON_INTERPRETATION_CSV}")
print(f"Summary JSON: {COMPARISON_SUMMARY_JSON}")

required_outputs = [
    COMBINED_PATIENT_PREDICTIONS_CSV,
    COMBINED_BOOTSTRAP_CI_CSV,
    PAIRED_COMPARISON_CSV,
    COMPARISON_INTERPRETATION_CSV,
    COMPARISON_SUMMARY_JSON,
    PARTIAL_BOOTSTRAP_CI_CSV,
    PARTIAL_PAIRED_COMPARISON_CSV,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 6 outputs: {missing_outputs}")

if len(bootstrap_ci) != len(model_ids) * len(COMPARISON_METRICS):
    raise RuntimeError("Unexpected number of bootstrap CI rows.")

if len(paired_comparison) != len(comparison_pairs) * len(COMPARISON_METRICS):
    raise RuntimeError("Unexpected number of paired comparison rows.")

print("Status: PASS")

In [ ]:
# Cell 7: Generate updated full-image vs anatomy-constrained comparison figures

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

COMPARISON_ROOT = RESULTS_10_DIR / "anatomy_vs_full_image_comparison"
FIGURE_ROOT = RESULTS_10_DIR / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

COMBINED_BOOTSTRAP_CI_CSV = (
    COMPARISON_ROOT
    / "notebook10_combined_patient_metric_bootstrap_ci.csv"
)

PAIRED_COMPARISON_CSV = (
    COMPARISON_ROOT
    / "notebook10_paired_anatomy_vs_full_image_comparison.csv"
)

COMPARISON_INTERPRETATION_CSV = (
    COMPARISON_ROOT
    / "notebook10_anatomy_vs_full_image_interpretation_flags.csv"
)

MAIN_COMPARISON_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.png"
)

MAIN_COMPARISON_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.pdf"
)

MAIN_COMPARISON_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.svg"
)

PRIMARY_SUMMARY_CSV = (
    RESULTS_10_DIR
    / "notebook10_primary_full_image_vs_anatomy_summary.csv"
)

FIGURE_MANIFEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_comparison_figure_manifest.csv"
)

FIGURE_SUMMARY_JSON = (
    RESULTS_10_DIR
    / "notebook10_comparison_figure_summary.json"
)

required_inputs = [
    COMBINED_BOOTSTRAP_CI_CSV,
    PAIRED_COMPARISON_CSV,
    COMPARISON_INTERPRETATION_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 7 inputs: {missing_inputs}")

bootstrap_ci = pd.read_csv(COMBINED_BOOTSTRAP_CI_CSV)
paired_comparison = pd.read_csv(PAIRED_COMPARISON_CSV)
interpretation_df = pd.read_csv(COMPARISON_INTERPRETATION_CSV)

available_model_ids = set(bootstrap_ci["model_id"].astype(str).unique())

DESIRED_MODEL_ORDER = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
]

MODEL_ORDER = [
    model_id
    for model_id in DESIRED_MODEL_ORDER
    if model_id in available_model_ids
]

missing_expected_models = [
    model_id
    for model_id in DESIRED_MODEL_ORDER
    if model_id not in available_model_ids
]

if missing_expected_models:
    raise RuntimeError(
        f"Expected models missing from bootstrap CI: {missing_expected_models}"
    )

MODEL_LABELS = {
    "full_image__EfficientNetB0": "Full image\nEffNet-B0",
    "full_image__MobileNetV3Large": "Full image\nMobileNetV3",
    "full_image__DeiTSmall": "Full image\nDeiT-Small",
    "anatomy__od_centered_crop_2p5x": "OD-centered\n2.5x",
    "anatomy__od_centered_crop_4p0x": "OD-centered\n4.0x",
    "anatomy__od_only_context_suppressed": "OD-only\ntrained",
    "anatomy__outside_od_only_trained_control": "Outside-OD\ntrained control",
}

MODEL_GROUPS = {
    "full_image__EfficientNetB0": "Full-image",
    "full_image__MobileNetV3Large": "Full-image",
    "full_image__DeiTSmall": "Full-image",
    "anatomy__od_centered_crop_2p5x": "Anatomy-constrained",
    "anatomy__od_centered_crop_4p0x": "Anatomy-constrained",
    "anatomy__od_only_context_suppressed": "Anatomy-constrained stress test",
    "anatomy__outside_od_only_trained_control": "Shortcut-control",
}

MODEL_COLORS = {
    "Full-image": "#4C78A8",
    "Anatomy-constrained": "#54A24B",
    "Anatomy-constrained stress test": "#F58518",
    "Shortcut-control": "#B279A2",
}

PRIMARY_REFERENCE = "full_image__EfficientNetB0"

ANATOMY_MODELS = [
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
]

ANATOMY_MODELS = [
    model_id
    for model_id in ANATOMY_MODELS
    if model_id in available_model_ids
]

PRIMARY_METRICS = [
    "auroc",
    "balanced_accuracy",
    "brier_score",
]

METRIC_LABELS = {
    "auroc": "Patient AUROC",
    "balanced_accuracy": "Patient balanced accuracy",
    "brier_score": "Patient Brier score",
}

def get_bootstrap_row(model_id, metric):
    block = bootstrap_ci[
        (bootstrap_ci["model_id"].astype(str) == model_id)
        & (bootstrap_ci["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(f"Bootstrap row not found uniquely: {model_id} | {metric}")

    return block.iloc[0]


def get_paired_row(reference_model_id, comparator_model_id, metric):
    block = paired_comparison[
        (paired_comparison["reference_model_id"].astype(str) == reference_model_id)
        & (paired_comparison["comparator_model_id"].astype(str) == comparator_model_id)
        & (paired_comparison["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(
            f"Paired row not found uniquely: {reference_model_id} | {comparator_model_id} | {metric}"
        )

    return block.iloc[0]


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )


def style_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="both", labelsize=9)


summary_rows = []

for model_id in MODEL_ORDER:
    for metric in PRIMARY_METRICS:
        row = get_bootstrap_row(model_id, metric)

        summary_rows.append(
            {
                "model_id": model_id,
                "model_label": MODEL_LABELS[model_id],
                "model_group": MODEL_GROUPS[model_id],
                "metric": metric,
                "metric_label": METRIC_LABELS[metric],
                "observed": float(row["observed"]),
                "ci95_low": float(row["ci95_low"]),
                "ci95_high": float(row["ci95_high"]),
                "n_valid_bootstrap": int(row["n_valid_bootstrap"]),
            }
        )

primary_summary = pd.DataFrame(summary_rows)
primary_summary.to_csv(PRIMARY_SUMMARY_CSV, index=False)

fig = plt.figure(figsize=(18, 10.5))

gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.0, 1.0],
    width_ratios=[1.0, 1.0, 1.0],
    hspace=0.50,
    wspace=0.36,
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

x = np.arange(len(MODEL_ORDER))

bar_colors = [
    MODEL_COLORS[MODEL_GROUPS[model_id]]
    for model_id in MODEL_ORDER
]

for ax_idx, metric in enumerate(PRIMARY_METRICS):
    ax = axes[ax_idx]

    values = []
    lower_errors = []
    upper_errors = []

    for model_id in MODEL_ORDER:
        row = get_bootstrap_row(model_id, metric)

        observed = float(row["observed"])
        ci_low = float(row["ci95_low"])
        ci_high = float(row["ci95_high"])

        values.append(observed)
        lower_errors.append(abs(observed - ci_low))
        upper_errors.append(abs(ci_high - observed))

    ax.bar(
        x,
        values,
        color=bar_colors,
        alpha=0.88,
    )

    ax.errorbar(
        x,
        values,
        yerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )

    ax.axvline(2.5, color="black", linewidth=0.8, linestyle="--", alpha=0.45)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [MODEL_LABELS[m] for m in MODEL_ORDER],
        rotation=35,
        ha="right",
    )

    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(f"{METRIC_LABELS[metric]} with 95% bootstrap CI")

    if metric in ["auroc", "balanced_accuracy"]:
        ax.set_ylim(0.75, 1.03)
    else:
        max_value = max(values)
        ax.set_ylim(0.0, max(max_value * 1.45, 0.03))

    style_axis(ax, grid_axis="y")
    add_panel_label(ax, chr(ord("A") + ax_idx))

delta_metrics = [
    "auroc",
    "balanced_accuracy",
    "brier_score",
]

for delta_idx, metric in enumerate(delta_metrics):
    ax = axes[3 + delta_idx]

    y_positions = np.arange(len(ANATOMY_MODELS))

    values = []
    lower_errors = []
    upper_errors = []
    colors = []

    for anatomy_model in ANATOMY_MODELS:
        row = get_paired_row(
            reference_model_id=PRIMARY_REFERENCE,
            comparator_model_id=anatomy_model,
            metric=metric,
        )

        delta = float(row["delta_comparator_minus_reference"])
        ci_low = float(row["delta_ci95_low"])
        ci_high = float(row["delta_ci95_high"])

        values.append(delta)
        lower_errors.append(abs(delta - ci_low))
        upper_errors.append(abs(ci_high - delta))
        colors.append(MODEL_COLORS[MODEL_GROUPS[anatomy_model]])

    ax.barh(
        y_positions,
        values,
        color=colors,
        alpha=0.88,
    )

    ax.errorbar(
        values,
        y_positions,
        xerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )

    ax.axvline(0, color="black", linewidth=1.0)

    ax.set_yticks(y_positions)
    ax.set_yticklabels([MODEL_LABELS[m] for m in ANATOMY_MODELS])

    ax.set_xlabel(
        f"Δ {METRIC_LABELS[metric]}\n"
        "(comparator − full-image EfficientNet-B0)"
    )

    ax.set_title("Paired delta vs full-image EfficientNet-B0")

    style_axis(ax, grid_axis="x")
    add_panel_label(ax, chr(ord("D") + delta_idx))

legend_handles = []

for group_name, color in MODEL_COLORS.items():
    if group_name in set(MODEL_GROUPS[m] for m in MODEL_ORDER):
        legend_handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                label=group_name,
                markerfacecolor=color,
                markersize=10,
            )
        )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=4,
    frameon=False,
    fontsize=10,
)

fig.suptitle(
    "Full-image versus anatomy-constrained and shortcut-control glaucoma classifiers",
    fontsize=15,
    fontweight="bold",
    y=1.055,
)

plt.savefig(MAIN_COMPARISON_FIGURE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(MAIN_COMPARISON_FIGURE_PDF, bbox_inches="tight")
plt.savefig(MAIN_COMPARISON_FIGURE_SVG, bbox_inches="tight")
plt.show()

figure_manifest = pd.DataFrame(
    [
        {
            "figure_name": "main_full_image_vs_anatomy_constrained_comparison",
            "format": "png",
            "figure_path": str(MAIN_COMPARISON_FIGURE_PNG),
            "exists": MAIN_COMPARISON_FIGURE_PNG.exists(),
        },
        {
            "figure_name": "main_full_image_vs_anatomy_constrained_comparison",
            "format": "pdf",
            "figure_path": str(MAIN_COMPARISON_FIGURE_PDF),
            "exists": MAIN_COMPARISON_FIGURE_PDF.exists(),
        },
        {
            "figure_name": "main_full_image_vs_anatomy_constrained_comparison",
            "format": "svg",
            "figure_path": str(MAIN_COMPARISON_FIGURE_SVG),
            "exists": MAIN_COMPARISON_FIGURE_SVG.exists(),
        },
    ]
)

figure_manifest.to_csv(FIGURE_MANIFEST_CSV, index=False)

primary_delta_rows = []

for anatomy_model in ANATOMY_MODELS:
    for metric in PRIMARY_METRICS:
        row = get_paired_row(
            reference_model_id=PRIMARY_REFERENCE,
            comparator_model_id=anatomy_model,
            metric=metric,
        )

        primary_delta_rows.append(
            {
                "reference_model_id": PRIMARY_REFERENCE,
                "comparator_model_id": anatomy_model,
                "metric": metric,
                "reference_observed": float(row["reference_observed"]),
                "comparator_observed": float(row["comparator_observed"]),
                "delta_comparator_minus_reference": float(row["delta_comparator_minus_reference"]),
                "delta_ci95_low": float(row["delta_ci95_low"]),
                "delta_ci95_high": float(row["delta_ci95_high"]),
                "paired_bootstrap_p_two_sided": float(row["paired_bootstrap_p_two_sided"]),
            }
        )

primary_delta_df = pd.DataFrame(primary_delta_rows)

figure_summary = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "cell": "07_generate_full_image_vs_anatomy_comparison_figures_with_outside_od_control",
    "status": "PASS",
    "main_figure_png": str(MAIN_COMPARISON_FIGURE_PNG),
    "main_figure_pdf": str(MAIN_COMPARISON_FIGURE_PDF),
    "main_figure_svg": str(MAIN_COMPARISON_FIGURE_SVG),
    "figure_manifest_csv": str(FIGURE_MANIFEST_CSV),
    "primary_summary_csv": str(PRIMARY_SUMMARY_CSV),
    "models_included": MODEL_ORDER,
    "anatomy_comparators_vs_primary_reference": ANATOMY_MODELS,
    "primary_reference_model": PRIMARY_REFERENCE,
    "primary_interpretation": (
        "Anatomy-constrained classifiers preserve patient-level discrimination comparable to full-image "
        "classifiers. The outside-OD trained control also performs strongly, supporting the presence of "
        "extra-papillary or dataset-specific predictive signal outside the optic disc. It should be interpreted "
        "as a shortcut-control model, not as a clinically preferred final model."
    ),
    "primary_deltas_vs_full_image_efficientnetb0": primary_delta_df.to_dict(orient="records"),
}

with open(FIGURE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(figure_summary, f, indent=2)

print("Models included in updated figure:")
print("\n".join(MODEL_ORDER))

print("\nAnatomy comparators included in paired delta panels:")
print("\n".join(ANATOMY_MODELS))

print("\nPrimary comparison summary:")
print(
    primary_summary[
        [
            "model_id",
            "metric",
            "observed",
            "ci95_low",
            "ci95_high",
            "n_valid_bootstrap",
        ]
    ].to_string(index=False)
)

print("\nPrimary deltas vs full-image EfficientNetB0:")
print(primary_delta_df.to_string(index=False))

print("\nGenerated figures:")
print(figure_manifest.to_string(index=False))

print(f"\nPrimary summary CSV: {PRIMARY_SUMMARY_CSV}")
print(f"Figure manifest CSV: {FIGURE_MANIFEST_CSV}")
print(f"Figure summary JSON: {FIGURE_SUMMARY_JSON}")

if "anatomy__outside_od_only_trained_control" not in set(primary_summary["model_id"].astype(str)):
    raise RuntimeError("Updated Cell 7 did not include outside-OD trained control in summary.")

if "anatomy__outside_od_only_trained_control" not in set(primary_delta_df["comparator_model_id"].astype(str)):
    raise RuntimeError("Updated Cell 7 did not include outside-OD trained control in paired deltas.")

if not figure_manifest["exists"].astype(bool).all():
    missing_figures = figure_manifest.loc[
        ~figure_manifest["exists"].astype(bool),
        "figure_path",
    ].tolist()
    raise RuntimeError(f"Missing generated figures: {missing_figures}")

required_outputs = [
    MAIN_COMPARISON_FIGURE_PNG,
    MAIN_COMPARISON_FIGURE_PDF,
    MAIN_COMPARISON_FIGURE_SVG,
    PRIMARY_SUMMARY_CSV,
    FIGURE_MANIFEST_CSV,
    FIGURE_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 7 outputs: {missing_outputs}")

print("Status: PASS")

In [ ]:
# Cell 8: Add outside-OD-only trained-control inputs

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import time
import shutil

OUTSIDE_OD_EXPERIMENT_ID = "outside_od_only_trained_control"
OUTSIDE_OD_DILATION_PIXELS = 7
OUTSIDE_OD_INPUT_SIZE = 384

LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

ANATOMY_INPUT_ROOT = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "strict_anatomical_inputs_384"
)

ANATOMY_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

OUTSIDE_OD_INPUT_ROOT = (
    ANATOMY_INPUT_ROOT
    / OUTSIDE_OD_EXPERIMENT_ID
)

OUTSIDE_OD_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split_name in ["train", "val", "test"]:
    (OUTSIDE_OD_INPUT_ROOT / split_name).mkdir(parents=True, exist_ok=True)

RUNTIME_INPUT_CACHE_ROOT = Path("/content/hygd_anatomy_constrained_inputs_384")
RUNTIME_INPUT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_INPUT_MANIFEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_anatomy_constrained_input_manifest.csv"
)

OUTSIDE_OD_INPUT_MANIFEST_CSV = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "notebook10_outside_od_only_trained_control_input_manifest.csv"
)

UPDATED_RUNTIME_INPUT_MANIFEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_anatomy_constrained_input_manifest_with_outside_od.csv"
)

OUTSIDE_OD_INPUT_SUMMARY_CSV = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_input_summary.csv"
)

OUTSIDE_OD_ERROR_LOG_CSV = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_error_log.csv"
)

OUTSIDE_OD_VISUAL_AUDIT_CASES_CSV = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_visual_audit_cases.csv"
)

OUTSIDE_OD_VISUAL_AUDIT_PANEL_PNG = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_visual_audit_panel.png"
)

OUTSIDE_OD_VISUAL_AUDIT_PANEL_PDF = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_visual_audit_panel.pdf"
)

OUTSIDE_OD_BUILD_SUMMARY_JSON = (
    RESULTS_10_DIR
    / "notebook10_outside_od_only_trained_control_build_summary.json"
)

PARTIAL_OUTSIDE_OD_MANIFEST_CSV = (
    ANATOMY_CONSTRAINED_DATA_DIR
    / "PARTIAL_notebook10_outside_od_only_trained_control_input_manifest.csv"
)

PARTIAL_OUTSIDE_OD_ERROR_LOG_CSV = (
    RESULTS_10_DIR
    / "PARTIAL_notebook10_outside_od_only_trained_control_error_log.csv"
)

PARTIAL_SAVE_EVERY = 50


def candidate_image_paths(image_path):
    image_path = str(image_path)
    candidates = []

    local_candidate = LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name
    candidates.append(str(local_candidate))
    candidates.append(image_path)

    seen = set()
    unique_candidates = []

    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)

    return unique_candidates


def read_rgb_image_robust(image_path, n_attempts=3, sleep_sec=0.25):
    last_error = ""

    for candidate in candidate_image_paths(image_path):
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    return image_rgb, candidate

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read image from candidates for {image_path}. Last error: {last_error}"
    )


def read_mask_robust(mask_path, n_attempts=3, sleep_sec=0.25):
    last_error = ""

    for _ in range(n_attempts):
        try:
            mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

            if mask_raw is not None:
                return mask_raw

            last_error = f"cv2.imread returned None for {mask_path}"

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(sleep_sec)

    raise FileNotFoundError(f"Could not read mask: {mask_path}. Last error: {last_error}")


def parse_grayscale_od_oc_mask(gray):
    gray = np.asarray(gray)
    positive_values = sorted([int(v) for v in np.unique(gray) if int(v) > 0])

    if len(positive_values) == 0:
        od_mask = np.zeros(gray.shape, dtype=bool)
        oc_mask = np.zeros(gray.shape, dtype=bool)

    elif len(positive_values) == 1:
        od_mask = gray == positive_values[0]
        oc_mask = np.zeros(gray.shape, dtype=bool)

    else:
        oc_value = positive_values[-1]
        od_mask = gray > 0
        oc_mask = gray == oc_value

    od_mask = od_mask | oc_mask

    return od_mask.astype(bool), oc_mask.astype(bool)


def parse_od_oc_mask(mask_raw):
    if mask_raw.ndim == 3:
        mask_rgb = cv2.cvtColor(mask_raw[:, :, :3], cv2.COLOR_BGR2RGB)
        flat = mask_rgb.reshape(-1, 3)
        unique_colors = np.unique(flat, axis=0)

        non_black_colors = [
            tuple(color.tolist())
            for color in unique_colors
            if int(color.sum()) > 0
        ]

        if len(non_black_colors) >= 2:
            color_areas = []

            for color in non_black_colors:
                color_array = np.array(color, dtype=np.uint8)
                color_mask = np.all(mask_rgb == color_array, axis=-1)
                color_areas.append((color, int(color_mask.sum())))

            color_areas = sorted(color_areas, key=lambda x: x[1], reverse=True)

            od_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)
            oc_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)

            largest_area = color_areas[0][1]

            for color, area in color_areas:
                color_array = np.array(color, dtype=np.uint8)
                current_mask = np.all(mask_rgb == color_array, axis=-1)

                if area == largest_area:
                    od_mask = od_mask | current_mask
                else:
                    oc_mask = oc_mask | current_mask
                    od_mask = od_mask | current_mask

        else:
            gray = cv2.cvtColor(mask_rgb, cv2.COLOR_RGB2GRAY)
            od_mask, oc_mask = parse_grayscale_od_oc_mask(gray)

    else:
        od_mask, oc_mask = parse_grayscale_od_oc_mask(mask_raw)

    od_mask = od_mask | oc_mask
    rim_mask = od_mask & (~oc_mask)

    return {
        "od_mask": od_mask.astype(bool),
        "oc_mask": oc_mask.astype(bool),
        "rim_mask": rim_mask.astype(bool),
        "od_plus_oc_mask": od_mask.astype(bool),
    }


def resize_mask_to_shape(mask, target_shape):
    target_h, target_w = target_shape

    resized = cv2.resize(
        mask.astype(np.uint8),
        (target_w, target_h),
        interpolation=cv2.INTER_NEAREST,
    ).astype(bool)

    return resized


def compute_fov_mask(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    fov = gray > 10

    kernel = np.ones((7, 7), np.uint8)
    fov = cv2.morphologyEx(fov.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    fov = cv2.morphologyEx(fov.astype(np.uint8), cv2.MORPH_OPEN, kernel)

    return fov.astype(bool)


def mean_color_from_region(image_rgb, region_mask):
    region_mask = region_mask.astype(bool)

    if int(region_mask.sum()) == 0:
        return image_rgb.reshape(-1, 3).mean(axis=0).astype(np.float32)

    return image_rgb[region_mask].reshape(-1, 3).mean(axis=0).astype(np.float32)


def get_mask_geometry(mask):
    ys, xs = np.where(mask.astype(bool))

    if len(xs) == 0:
        raise RuntimeError("Empty anatomical mask.")

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    center_x = float(xs.mean())
    center_y = float(ys.mean())

    area = int(mask.sum())
    equivalent_diameter = float(2.0 * np.sqrt(area / np.pi))

    return {
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "center_x": center_x,
        "center_y": center_y,
        "area": area,
        "equivalent_diameter": equivalent_diameter,
    }


def dilate_binary_mask(mask, dilation_pixels):
    if dilation_pixels <= 0:
        return mask.astype(bool)

    kernel_size = int(2 * dilation_pixels + 1)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size),
    )

    dilated = cv2.dilate(
        mask.astype(np.uint8),
        kernel,
        iterations=1,
    )

    return dilated.astype(bool)


def make_od_oc_overlay(image_rgb, od_mask, oc_mask, alpha=0.45):
    od_mask = od_mask.astype(bool)
    oc_mask = oc_mask.astype(bool)
    rim_mask = od_mask & (~oc_mask)

    overlay = image_rgb.copy().astype(np.float32)
    color_layer = np.zeros_like(overlay)

    color_layer[rim_mask] = np.array([255, 180, 0], dtype=np.float32)
    color_layer[oc_mask] = np.array([0, 180, 255], dtype=np.float32)

    active_mask = od_mask | oc_mask

    overlay[active_mask] = (
        (1.0 - alpha) * overlay[active_mask]
        + alpha * color_layer[active_mask]
    )

    return np.clip(overlay, 0, 255).astype(np.uint8)


def save_rgb_png(image_rgb, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(output_path), image_bgr)

    if not ok:
        raise RuntimeError(f"cv2.imwrite failed for {output_path}")

    return str(output_path)


def safe_copy_to_runtime_cache(source_path, local_path):
    source_path = Path(str(source_path))
    local_path = Path(str(local_path))
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        return str(local_path), "already_cached"

    shutil.copy2(str(source_path), str(local_path))

    return str(local_path), "copied"


def build_outside_od_only_image(
    image_rgb,
    od_mask_original_shape,
    output_size=OUTSIDE_OD_INPUT_SIZE,
    dilation_pixels=OUTSIDE_OD_DILATION_PIXELS,
):
    resized_rgb = cv2.resize(
        image_rgb,
        (output_size, output_size),
        interpolation=cv2.INTER_AREA,
    )

    od_mask_384 = resize_mask_to_shape(
        od_mask_original_shape,
        target_shape=(output_size, output_size),
    )

    fov_mask = compute_fov_mask(resized_rgb)

    dilated_od_mask = dilate_binary_mask(
        od_mask_384,
        dilation_pixels=dilation_pixels,
    )

    dilated_od_mask = dilated_od_mask & fov_mask
    outside_od_fov = fov_mask & (~dilated_od_mask)

    outside_mean = mean_color_from_region(
        resized_rgb,
        outside_od_fov,
    )

    output = resized_rgb.copy().astype(np.float32)
    output[dilated_od_mask] = outside_mean

    non_fov_mask = ~fov_mask
    output[non_fov_mask] = resized_rgb[non_fov_mask]

    output = np.clip(output, 0, 255).astype(np.uint8)

    geometry = get_mask_geometry(od_mask_384)

    metadata = {
        **geometry,
        "input_size": int(output_size),
        "outside_od_dilation_pixels": int(dilation_pixels),
        "removed_od_pixels": int(dilated_od_mask.sum()),
        "preserved_outside_od_fov_pixels": int(outside_od_fov.sum()),
        "fov_pixels": int(fov_mask.sum()),
        "removed_od_fraction_of_fov": float(
            dilated_od_mask.sum() / max(fov_mask.sum(), 1)
        ),
    }

    return output, metadata


outside_manifest_rows = []
outside_error_rows = []

for row_idx, row in tqdm(
    training_ready_manifest.iterrows(),
    total=len(training_ready_manifest),
    desc="Building outside-OD-only trained-control inputs",
):
    sample_id = str(row["sample_id"])

    try:
        image_rgb, resolved_image_path = read_rgb_image_robust(
            row["image_path_for_inference"]
        )

        mask_raw = read_mask_robust(row["mask_512_path"])
        masks_512 = parse_od_oc_mask(mask_raw)

        od_mask_original = resize_mask_to_shape(
            masks_512["od_mask"],
            target_shape=image_rgb.shape[:2],
        )

        oc_mask_original = resize_mask_to_shape(
            masks_512["oc_mask"],
            target_shape=image_rgb.shape[:2],
        )

        od_mask_original = od_mask_original | oc_mask_original

        if int(od_mask_original.sum()) == 0:
            raise RuntimeError("Empty OD mask after resizing to original image.")

        outside_od_rgb, metadata = build_outside_od_only_image(
            image_rgb=image_rgb,
            od_mask_original_shape=od_mask_original,
            output_size=OUTSIDE_OD_INPUT_SIZE,
            dilation_pixels=OUTSIDE_OD_DILATION_PIXELS,
        )

        output_path = (
            OUTSIDE_OD_INPUT_ROOT
            / str(row["glaucoma_split"])
            / f"{sample_id}_{OUTSIDE_OD_EXPERIMENT_ID}_384.png"
        )

        saved_path = save_rgb_png(
            outside_od_rgb,
            output_path,
        )

        outside_manifest_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": str(row["patient_id"]),
                "gon_binary": int(row["gon_binary"]),
                "gon_label": str(row["gon_label"]),
                "glaucoma_split": str(row["glaucoma_split"]),
                "quality_score": float(row["quality_score"]),
                "area_cdr_proxy": float(row["area_cdr_proxy"]) if pd.notna(row["area_cdr_proxy"]) else np.nan,
                "experiment_id": OUTSIDE_OD_EXPERIMENT_ID,
                "training_role": "outside_od_trained_shortcut_control",
                "candidate_model": CANDIDATE_TIMM_MODEL,
                "input_size": int(OUTSIDE_OD_INPUT_SIZE),
                "crop_scale": np.nan,
                "context_policy": "outside_od_context_preserved_od_removed",
                "background_policy": "dilated_od_replaced_by_outside_fov_mean",
                "constrained_image_path": saved_path,
                "source_image_path": str(row["image_path_for_inference"]),
                "resolved_image_path": str(resolved_image_path),
                "source_mask_path": str(row["mask_512_path"]),
                "od_area_original_px": int(od_mask_original.sum()),
                "oc_area_original_px": int(oc_mask_original.sum()),
                **metadata,
            }
        )

    except Exception as exc:
        outside_error_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": str(row.get("patient_id", "")),
                "glaucoma_split": str(row.get("glaucoma_split", "")),
                "error": repr(exc),
            }
        )

    if (row_idx + 1) % PARTIAL_SAVE_EVERY == 0:
        pd.DataFrame(outside_manifest_rows).to_csv(
            PARTIAL_OUTSIDE_OD_MANIFEST_CSV,
            index=False,
        )

        pd.DataFrame(outside_error_rows).to_csv(
            PARTIAL_OUTSIDE_OD_ERROR_LOG_CSV,
            index=False,
        )

outside_manifest = pd.DataFrame(outside_manifest_rows)
outside_error_log = pd.DataFrame(outside_error_rows)

outside_manifest.to_csv(OUTSIDE_OD_INPUT_MANIFEST_CSV, index=False)
outside_error_log.to_csv(OUTSIDE_OD_ERROR_LOG_CSV, index=False)

outside_summary = (
    outside_manifest
    .groupby(["experiment_id", "glaucoma_split"])
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_gon_positive_images=("gon_binary", "sum"),
        mean_quality_score=("quality_score", "mean"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
        mean_removed_od_pixels=("removed_od_pixels", "mean"),
        mean_preserved_outside_od_fov_pixels=("preserved_outside_od_fov_pixels", "mean"),
        mean_removed_od_fraction_of_fov=("removed_od_fraction_of_fov", "mean"),
    )
    .reset_index()
)

outside_summary["n_gon_negative_images"] = (
    outside_summary["n_images"]
    - outside_summary["n_gon_positive_images"]
)

outside_summary.to_csv(OUTSIDE_OD_INPUT_SUMMARY_CSV, index=False)

if not RUNTIME_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing runtime manifest from Cell 3: {RUNTIME_INPUT_MANIFEST_CSV}")

current_runtime_manifest = pd.read_csv(RUNTIME_INPUT_MANIFEST_CSV)

outside_runtime_rows = []

for _, row in outside_manifest.iterrows():
    source_path = Path(str(row["constrained_image_path"]))

    local_path = (
        RUNTIME_INPUT_CACHE_ROOT
        / OUTSIDE_OD_EXPERIMENT_ID
        / str(row["glaucoma_split"])
        / source_path.name
    )

    runtime_path, cache_status = safe_copy_to_runtime_cache(
        source_path=source_path,
        local_path=local_path,
    )

    output_row = row.to_dict()
    output_row["runtime_image_path"] = str(runtime_path)
    output_row["runtime_exists"] = bool(Path(runtime_path).exists())

    outside_runtime_rows.append(output_row)

outside_runtime_manifest = pd.DataFrame(outside_runtime_rows)

updated_runtime_manifest = pd.concat(
    [
        current_runtime_manifest,
        outside_runtime_manifest,
    ],
    axis=0,
    ignore_index=True,
)

updated_runtime_manifest.to_csv(UPDATED_RUNTIME_INPUT_MANIFEST_CSV, index=False)

input_manifest = updated_runtime_manifest.copy()

visual_cases = []

for split_name in ["train", "val", "test"]:
    for label_value in [1, 0]:
        block = (
            training_ready_manifest[
                (training_ready_manifest["glaucoma_split"] == split_name)
                & (training_ready_manifest["gon_binary"] == label_value)
            ]
            .copy()
            .sort_values(["area_cdr_proxy", "quality_score", "sample_id"])
            .reset_index(drop=True)
        )

        if len(block) == 0:
            continue

        selected_idx = len(block) // 2
        visual_cases.append(block.iloc[selected_idx].to_dict())

visual_cases_df = pd.DataFrame(visual_cases)
visual_cases_df["sample_id"] = visual_cases_df["sample_id"].astype(str)
visual_cases_df.to_csv(OUTSIDE_OD_VISUAL_AUDIT_CASES_CSV, index=False)

n_rows = len(visual_cases_df)

fig, axes = plt.subplots(
    n_rows,
    3,
    figsize=(12, max(3.2 * n_rows, 5)),
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, case_row) in enumerate(visual_cases_df.iterrows()):
    sample_id = str(case_row["sample_id"])

    image_rgb, _ = read_rgb_image_robust(case_row["image_path_for_inference"])
    mask_raw = read_mask_robust(case_row["mask_512_path"])
    masks_512 = parse_od_oc_mask(mask_raw)

    image_384 = cv2.resize(
        image_rgb,
        (OUTSIDE_OD_INPUT_SIZE, OUTSIDE_OD_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    od_mask_384 = resize_mask_to_shape(
        masks_512["od_mask"],
        target_shape=(OUTSIDE_OD_INPUT_SIZE, OUTSIDE_OD_INPUT_SIZE),
    )

    oc_mask_384 = resize_mask_to_shape(
        masks_512["oc_mask"],
        target_shape=(OUTSIDE_OD_INPUT_SIZE, OUTSIDE_OD_INPUT_SIZE),
    )

    overlay_rgb = make_od_oc_overlay(
        image_rgb=image_384,
        od_mask=od_mask_384,
        oc_mask=oc_mask_384,
    )

    outside_path = (
        OUTSIDE_OD_INPUT_ROOT
        / str(case_row["glaucoma_split"])
        / f"{sample_id}_{OUTSIDE_OD_EXPERIMENT_ID}_384.png"
    )

    outside_rgb, _ = read_rgb_image_robust(outside_path)

    dilated_od_mask = dilate_binary_mask(
        od_mask_384 | oc_mask_384,
        dilation_pixels=OUTSIDE_OD_DILATION_PIXELS,
    )

    mask_display = image_384.copy()
    mask_display[dilated_od_mask] = np.array([255, 180, 0], dtype=np.uint8)

    display_images = [
        overlay_rgb,
        mask_display,
        outside_rgb,
    ]

    titles = [
        "Original + OD/OC",
        "Dilated OD removal mask",
        "Outside-OD trained input",
    ]

    row_label = (
        f"{sample_id} | {case_row['glaucoma_split']} | {case_row['gon_label']} | "
        f"Q={float(case_row['quality_score']):.2f} | "
        f"CDR={float(case_row['area_cdr_proxy']):.3f}"
    )

    for col_idx, image in enumerate(display_images):
        ax = axes[row_idx, col_idx]
        ax.imshow(image)
        ax.axis("off")

        if row_idx == 0:
            ax.set_title(titles[col_idx], fontsize=11)

        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=8)

fig.suptitle(
    "Notebook 10 visual audit: outside-OD-only trained-control inputs",
    fontsize=14,
    fontweight="bold",
    y=0.995,
)

plt.tight_layout()
plt.savefig(OUTSIDE_OD_VISUAL_AUDIT_PANEL_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUTSIDE_OD_VISUAL_AUDIT_PANEL_PDF, bbox_inches="tight")
plt.show()

build_summary = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "cell": "08_add_outside_od_only_trained_control_inputs",
    "status": "PASS",
    "experiment_id": OUTSIDE_OD_EXPERIMENT_ID,
    "input_size": int(OUTSIDE_OD_INPUT_SIZE),
    "outside_od_dilation_pixels": int(OUTSIDE_OD_DILATION_PIXELS),
    "n_training_ready_strict_images": int(len(training_ready_manifest)),
    "n_manifest_rows": int(len(outside_manifest)),
    "n_error_rows": int(len(outside_error_log)),
    "updated_runtime_manifest_rows": int(len(updated_runtime_manifest)),
    "outside_od_input_manifest_csv": str(OUTSIDE_OD_INPUT_MANIFEST_CSV),
    "updated_runtime_input_manifest_csv": str(UPDATED_RUNTIME_INPUT_MANIFEST_CSV),
    "outside_od_input_summary_csv": str(OUTSIDE_OD_INPUT_SUMMARY_CSV),
    "outside_od_error_log_csv": str(OUTSIDE_OD_ERROR_LOG_CSV),
    "visual_audit_cases_csv": str(OUTSIDE_OD_VISUAL_AUDIT_CASES_CSV),
    "visual_audit_panel_png": str(OUTSIDE_OD_VISUAL_AUDIT_PANEL_PNG),
    "visual_audit_panel_pdf": str(OUTSIDE_OD_VISUAL_AUDIT_PANEL_PDF),
}

with open(OUTSIDE_OD_BUILD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(build_summary, f, indent=2)

print("Outside-OD trained-control input summary:")
print(outside_summary.to_string(index=False))

print("\nUpdated runtime manifest experiment counts:")
print(
    updated_runtime_manifest
    .groupby(["experiment_id", "glaucoma_split"])
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_positive=("gon_binary", "sum"),
    )
    .reset_index()
    .to_string(index=False)
)

print(f"\nOutside-OD input manifest: {OUTSIDE_OD_INPUT_MANIFEST_CSV}")
print(f"Updated runtime input manifest: {UPDATED_RUNTIME_INPUT_MANIFEST_CSV}")
print(f"Outside-OD summary CSV: {OUTSIDE_OD_INPUT_SUMMARY_CSV}")
print(f"Outside-OD error log CSV: {OUTSIDE_OD_ERROR_LOG_CSV}")
print(f"Visual audit panel PNG: {OUTSIDE_OD_VISUAL_AUDIT_PANEL_PNG}")
print(f"Visual audit panel PDF: {OUTSIDE_OD_VISUAL_AUDIT_PANEL_PDF}")
print(f"Build summary JSON: {OUTSIDE_OD_BUILD_SUMMARY_JSON}")

if len(outside_error_log) > 0:
    print(outside_error_log.head(20).to_string(index=False))
    raise RuntimeError(f"Outside-OD input construction failed for {len(outside_error_log)} cases.")

if len(outside_manifest) != len(training_ready_manifest):
    raise RuntimeError(
        f"Unexpected outside-OD manifest rows: {len(outside_manifest)} != {len(training_ready_manifest)}"
    )

if not outside_runtime_manifest["runtime_exists"].astype(bool).all():
    missing_runtime = int((~outside_runtime_manifest["runtime_exists"].astype(bool)).sum())
    raise RuntimeError(f"Missing outside-OD runtime cached files: {missing_runtime}")

print("Status: PASS")

In [ ]:
# Cell 9: Train only the outside-OD-only trained-control classifier

OUTSIDE_OD_EXPERIMENT_ID = "outside_od_only_trained_control"

UPDATED_RUNTIME_INPUT_MANIFEST_CSV = (
    RESULTS_10_DIR
    / "notebook10_runtime_anatomy_constrained_input_manifest_with_outside_od.csv"
)

TRAINING_PROGRESS_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_training_progress.csv"
VALIDATION_RANKING_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_validation_ranking.csv"
VALIDATION_SELECTED_MODEL_JSON = RESULTS_10_DIR / "notebook10_validation_selected_anatomy_constrained_model.json"

if not UPDATED_RUNTIME_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing updated runtime manifest: {UPDATED_RUNTIME_INPUT_MANIFEST_CSV}")

required_runtime_objects = [
    "train_one_experiment",
    "make_dataloaders_for_experiment",
    "create_anatomy_classifier",
    "compute_class_loss_weights",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Missing training functions from previous cells. "
        f"Run Cell 3 and Cell 4 definitions first. Missing: {missing_runtime_objects}"
    )

input_manifest = pd.read_csv(UPDATED_RUNTIME_INPUT_MANIFEST_CSV)

input_manifest["sample_id"] = input_manifest["sample_id"].astype(str)
input_manifest["patient_id"] = input_manifest["patient_id"].astype(str)
input_manifest["experiment_id"] = input_manifest["experiment_id"].astype(str)
input_manifest["glaucoma_split"] = input_manifest["glaucoma_split"].astype(str)
input_manifest["runtime_image_path"] = input_manifest["runtime_image_path"].astype(str)
input_manifest["gon_binary"] = pd.to_numeric(input_manifest["gon_binary"], errors="coerce").astype(int)

outside_block = input_manifest[
    input_manifest["experiment_id"] == OUTSIDE_OD_EXPERIMENT_ID
].copy()

if len(outside_block) == 0:
    raise RuntimeError(f"No rows found for {OUTSIDE_OD_EXPERIMENT_ID} in updated runtime manifest.")

outside_split_summary = (
    outside_block
    .groupby("glaucoma_split")
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_positive=("gon_binary", "sum"),
    )
    .reset_index()
)

outside_split_summary["n_negative"] = (
    outside_split_summary["n_images"]
    - outside_split_summary["n_positive"]
)

expected_splits = {"train", "val", "test"}

if not expected_splits.issubset(set(outside_split_summary["glaucoma_split"].astype(str))):
    raise RuntimeError("Outside-OD experiment does not contain train/val/test splits.")

print("Training only outside-OD trained-control experiment")
print(f"Experiment: {OUTSIDE_OD_EXPERIMENT_ID}")
print(f"Device: {DEVICE}")
print("\nOutside-OD split summary:")
print(outside_split_summary.to_string(index=False))

result = train_one_experiment(OUTSIDE_OD_EXPERIMENT_ID)

print("\nOutside-OD training result:")
print(pd.DataFrame([result]).to_string(index=False))

# Update training progress CSV by merging previous progress with the new experiment.
if TRAINING_PROGRESS_CSV.exists():
    previous_progress = pd.read_csv(TRAINING_PROGRESS_CSV)
else:
    previous_progress = pd.DataFrame([])

new_progress = pd.DataFrame([result])

if len(previous_progress) > 0:
    updated_progress = pd.concat(
        [
            previous_progress[
                previous_progress["experiment_id"].astype(str) != OUTSIDE_OD_EXPERIMENT_ID
            ],
            new_progress,
        ],
        axis=0,
        ignore_index=True,
    )
else:
    updated_progress = new_progress.copy()

updated_progress.to_csv(TRAINING_PROGRESS_CSV, index=False)

# Rebuild validation ranking including all completed anatomy-constrained experiments.
result_frames = []

for _, progress_row in updated_progress.iterrows():
    result_csv = Path(str(progress_row["result_csv"]))

    if not result_csv.exists():
        raise RuntimeError(f"Missing result CSV: {result_csv}")

    result_frames.append(pd.read_csv(result_csv))

consolidated_results = pd.concat(result_frames, axis=0, ignore_index=True)

required_ranking_cols = [
    "experiment_id",
    "candidate_model",
    "best_epoch_one_indexed",
    "epochs_ran",
    "elapsed_min",
    "trainable_parameters",
    "best_patient_val_auroc",
    "best_patient_val_auprc",
    "best_patient_val_balanced_accuracy",
    "best_patient_val_brier_score",
    "best_patient_val_threshold",
    "best_patient_val_sensitivity",
    "best_patient_val_specificity",
    "best_image_val_auroc",
    "best_image_val_balanced_accuracy",
    "best_checkpoint_path",
    "training_log_csv",
    "val_image_predictions_csv",
    "val_patient_predictions_csv",
]

missing_cols = [
    col for col in required_ranking_cols
    if col not in consolidated_results.columns
]

if missing_cols:
    raise ValueError(f"Missing result columns for validation ranking: {missing_cols}")

consolidated_results["validation_selection_score"] = (
    0.45 * consolidated_results["best_patient_val_auroc"].astype(float)
    + 0.25 * consolidated_results["best_patient_val_auprc"].astype(float)
    + 0.20 * consolidated_results["best_patient_val_balanced_accuracy"].astype(float)
    - 0.10 * consolidated_results["best_patient_val_brier_score"].astype(float)
)

validation_ranking = consolidated_results.sort_values(
    [
        "validation_selection_score",
        "best_patient_val_auroc",
        "best_patient_val_brier_score",
    ],
    ascending=[False, False, True],
).reset_index(drop=True)

validation_ranking["validation_rank"] = np.arange(1, len(validation_ranking) + 1)

validation_ranking = validation_ranking[
    [
        "validation_rank",
        "experiment_id",
        "candidate_model",
        "best_epoch_one_indexed",
        "epochs_ran",
        "elapsed_min",
        "trainable_parameters",
        "best_patient_val_auroc",
        "best_patient_val_auprc",
        "best_patient_val_balanced_accuracy",
        "best_patient_val_brier_score",
        "best_patient_val_threshold",
        "best_patient_val_sensitivity",
        "best_patient_val_specificity",
        "best_image_val_auroc",
        "best_image_val_balanced_accuracy",
        "validation_selection_score",
        "best_checkpoint_path",
        "training_log_csv",
        "val_image_predictions_csv",
        "val_patient_predictions_csv",
    ]
]

validation_ranking.to_csv(VALIDATION_RANKING_CSV, index=False)

selected_row = validation_ranking.iloc[0].to_dict()

selected_payload = {
    "selection_stage": "validation_only_anatomy_constrained_model_selection_with_outside_od_control",
    "selected_experiment_id": str(selected_row["experiment_id"]),
    "selected_candidate_model": str(selected_row["candidate_model"]),
    "selected_best_checkpoint": str(selected_row["best_checkpoint_path"]),
    "selected_best_epoch_one_indexed": int(selected_row["best_epoch_one_indexed"]),
    "selected_best_patient_val_auroc": float(selected_row["best_patient_val_auroc"]),
    "selected_best_patient_val_auprc": float(selected_row["best_patient_val_auprc"]),
    "selected_best_patient_val_balanced_accuracy": float(selected_row["best_patient_val_balanced_accuracy"]),
    "selected_best_patient_val_brier_score": float(selected_row["best_patient_val_brier_score"]),
    "selected_best_patient_val_threshold": float(selected_row["best_patient_val_threshold"]),
    "validation_selection_score": float(selected_row["validation_selection_score"]),
    "selection_score_formula": (
        "0.45*patient_AUROC + 0.25*patient_AUPRC + "
        "0.20*patient_balanced_accuracy - 0.10*patient_Brier"
    ),
    "test_set_policy": (
        "The locked test set is not used for model selection. "
        "The outside-OD trained-control is included as a shortcut-control experiment, "
        "not as the primary clinically plausible model."
    ),
    "primary_anatomically_parsimonious_model": "od_centered_crop_2p5x",
    "outside_od_control_model": OUTSIDE_OD_EXPERIMENT_ID,
    "validation_ranking_csv": str(VALIDATION_RANKING_CSV),
    "training_progress_csv": str(TRAINING_PROGRESS_CSV),
}

with open(VALIDATION_SELECTED_MODEL_JSON, "w", encoding="utf-8") as f:
    json.dump(selected_payload, f, indent=2)

print("\nUpdated anatomy-constrained validation ranking:")
print(
    validation_ranking[
        [
            "validation_rank",
            "experiment_id",
            "best_epoch_one_indexed",
            "best_patient_val_auroc",
            "best_patient_val_auprc",
            "best_patient_val_balanced_accuracy",
            "best_patient_val_brier_score",
            "validation_selection_score",
        ]
    ].to_string(index=False)
)

print("\nUpdated validation-selected anatomy-constrained classifier:")
print(json.dumps(selected_payload, indent=2))

print(f"\nTraining progress CSV: {TRAINING_PROGRESS_CSV}")
print(f"Validation ranking CSV: {VALIDATION_RANKING_CSV}")
print(f"Selected model JSON: {VALIDATION_SELECTED_MODEL_JSON}")

if OUTSIDE_OD_EXPERIMENT_ID not in set(validation_ranking["experiment_id"].astype(str)):
    raise RuntimeError("Outside-OD trained-control was not included in validation ranking.")

if len(validation_ranking) < 4:
    raise RuntimeError(f"Expected at least 4 anatomy experiments, found {len(validation_ranking)}.")

if not np.isfinite(validation_ranking["best_patient_val_auroc"]).all():
    raise RuntimeError("Non-finite validation AUROC found after adding outside-OD control.")

print("Status: PASS")

In [ ]:
# Cell 10: Final synthesis and status for Notebook 10

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

try:
    RESULTS_10_DIR
except NameError:
    RESULTS_10_DIR = PROJECT_ROOT / "results" / "10_anatomy_constrained_classifier_training"

try:
    ANATOMY_CONSTRAINED_DATA_DIR
except NameError:
    ANATOMY_CONSTRAINED_DATA_DIR = PROJECT_ROOT / "project_data" / "hygd_anatomy_constrained_classifier_inputs"

RESULTS_10_DIR.mkdir(parents=True, exist_ok=True)

ANATOMY_TEST_EVAL_ROOT = RESULTS_10_DIR / "anatomy_constrained_locked_test_evaluation"
COMPARISON_ROOT = RESULTS_10_DIR / "anatomy_vs_full_image_comparison"
FIGURE_ROOT = RESULTS_10_DIR / "figures"

SETUP_SUMMARY_JSON = RESULTS_10_DIR / "notebook10_setup_summary.json"
INPUT_BUILD_SUMMARY_JSON = RESULTS_10_DIR / "notebook10_anatomy_constrained_input_build_summary.json"
OUTSIDE_OD_BUILD_SUMMARY_JSON = RESULTS_10_DIR / "notebook10_outside_od_only_trained_control_build_summary.json"

TRAINING_PROGRESS_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_training_progress.csv"
VALIDATION_RANKING_CSV = RESULTS_10_DIR / "notebook10_anatomy_constrained_validation_ranking.csv"
VALIDATION_SELECTED_MODEL_JSON = RESULTS_10_DIR / "notebook10_validation_selected_anatomy_constrained_model.json"

TEST_PATIENT_METRICS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_patient_level_metrics.csv"
)

TEST_IMAGE_METRICS_CSV = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_test_image_level_metrics.csv"
)

TEST_EVALUATION_SUMMARY_JSON = (
    ANATOMY_TEST_EVAL_ROOT
    / "notebook10_anatomy_constrained_locked_test_evaluation_summary.json"
)

COMBINED_PATIENT_PREDICTIONS_CSV = (
    COMPARISON_ROOT
    / "notebook10_combined_full_image_and_anatomy_patient_predictions.csv"
)

COMBINED_BOOTSTRAP_CI_CSV = (
    COMPARISON_ROOT
    / "notebook10_combined_patient_metric_bootstrap_ci.csv"
)

PAIRED_COMPARISON_CSV = (
    COMPARISON_ROOT
    / "notebook10_paired_anatomy_vs_full_image_comparison.csv"
)

COMPARISON_INTERPRETATION_CSV = (
    COMPARISON_ROOT
    / "notebook10_anatomy_vs_full_image_interpretation_flags.csv"
)

COMPARISON_SUMMARY_JSON = (
    COMPARISON_ROOT
    / "notebook10_anatomy_vs_full_image_comparison_summary.json"
)

PRIMARY_SUMMARY_CSV = RESULTS_10_DIR / "notebook10_primary_full_image_vs_anatomy_summary.csv"
FIGURE_MANIFEST_CSV = RESULTS_10_DIR / "notebook10_comparison_figure_manifest.csv"
FIGURE_SUMMARY_JSON = RESULTS_10_DIR / "notebook10_comparison_figure_summary.json"

MAIN_COMPARISON_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.png"
)

MAIN_COMPARISON_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.pdf"
)

MAIN_COMPARISON_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook10_main_full_image_vs_anatomy_constrained_comparison.svg"
)

NOTEBOOK10_FINAL_SYNTHESIS_CSV = RESULTS_10_DIR / "notebook10_final_synthesis.csv"
NOTEBOOK10_FINAL_MODEL_ROLES_CSV = RESULTS_10_DIR / "notebook10_final_model_roles.csv"
NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV = RESULTS_10_DIR / "notebook10_final_primary_deltas_vs_full_image_efficientnetb0.csv"
NOTEBOOK10_FINAL_INTERPRETATION_CSV = RESULTS_10_DIR / "notebook10_final_interpretation.csv"
NOTEBOOK10_FINAL_CHECKS_CSV = RESULTS_10_DIR / "notebook10_final_status_checks.csv"
NOTEBOOK10_FINAL_STATUS_JSON = RESULTS_10_DIR / "notebook10_final_status.json"
NOTEBOOK10_FINAL_STATUS_TXT = RESULTS_10_DIR / "notebook10_FINAL_STATUS.txt"

required_inputs = [
    SETUP_SUMMARY_JSON,
    INPUT_BUILD_SUMMARY_JSON,
    OUTSIDE_OD_BUILD_SUMMARY_JSON,
    TRAINING_PROGRESS_CSV,
    VALIDATION_RANKING_CSV,
    VALIDATION_SELECTED_MODEL_JSON,
    TEST_PATIENT_METRICS_CSV,
    TEST_IMAGE_METRICS_CSV,
    TEST_EVALUATION_SUMMARY_JSON,
    COMBINED_PATIENT_PREDICTIONS_CSV,
    COMBINED_BOOTSTRAP_CI_CSV,
    PAIRED_COMPARISON_CSV,
    COMPARISON_INTERPRETATION_CSV,
    COMPARISON_SUMMARY_JSON,
    PRIMARY_SUMMARY_CSV,
    FIGURE_MANIFEST_CSV,
    FIGURE_SUMMARY_JSON,
    MAIN_COMPARISON_FIGURE_PNG,
    MAIN_COMPARISON_FIGURE_PDF,
    MAIN_COMPARISON_FIGURE_SVG,
]

input_audit = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "path": str(path),
            "exists": Path(path).exists(),
        }
        for path in required_inputs
    ]
)

missing_inputs = input_audit.loc[
    ~input_audit["exists"],
    "path",
].tolist()

if missing_inputs:
    print(input_audit.to_string(index=False))
    raise FileNotFoundError(f"Missing Notebook 10 final inputs: {missing_inputs}")

with open(SETUP_SUMMARY_JSON, "r", encoding="utf-8") as f:
    setup_summary = json.load(f)

with open(INPUT_BUILD_SUMMARY_JSON, "r", encoding="utf-8") as f:
    input_build_summary = json.load(f)

with open(OUTSIDE_OD_BUILD_SUMMARY_JSON, "r", encoding="utf-8") as f:
    outside_od_build_summary = json.load(f)

with open(VALIDATION_SELECTED_MODEL_JSON, "r", encoding="utf-8") as f:
    validation_selected_payload = json.load(f)

with open(TEST_EVALUATION_SUMMARY_JSON, "r", encoding="utf-8") as f:
    test_summary = json.load(f)

with open(COMPARISON_SUMMARY_JSON, "r", encoding="utf-8") as f:
    comparison_summary = json.load(f)

with open(FIGURE_SUMMARY_JSON, "r", encoding="utf-8") as f:
    figure_summary = json.load(f)

training_progress = pd.read_csv(TRAINING_PROGRESS_CSV)
validation_ranking = pd.read_csv(VALIDATION_RANKING_CSV)
test_patient_metrics = pd.read_csv(TEST_PATIENT_METRICS_CSV)
test_image_metrics = pd.read_csv(TEST_IMAGE_METRICS_CSV)
combined_predictions = pd.read_csv(COMBINED_PATIENT_PREDICTIONS_CSV)
bootstrap_ci = pd.read_csv(COMBINED_BOOTSTRAP_CI_CSV)
paired_comparison = pd.read_csv(PAIRED_COMPARISON_CSV)
comparison_interpretation = pd.read_csv(COMPARISON_INTERPRETATION_CSV)
primary_summary = pd.read_csv(PRIMARY_SUMMARY_CSV)
figure_manifest = pd.read_csv(FIGURE_MANIFEST_CSV)

EXPECTED_ANATOMY_EXPERIMENTS = [
    "od_centered_crop_2p5x",
    "od_centered_crop_4p0x",
    "od_only_context_suppressed",
    "outside_od_only_trained_control",
]

EXPECTED_MODEL_IDS = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
]

PRIMARY_FULL_IMAGE_REFERENCE = "full_image__EfficientNetB0"
PRIMARY_ANATOMICAL_MODEL = "anatomy__od_centered_crop_2p5x"
VALIDATION_SELECTED_ANATOMY_MODEL = f"anatomy__{validation_selected_payload['selected_experiment_id']}"
OUTSIDE_OD_CONTROL_MODEL = "anatomy__outside_od_only_trained_control"
OD_ONLY_STRESS_MODEL = "anatomy__od_only_context_suppressed"

MODEL_ROLES = {
    "full_image__EfficientNetB0": {
        "model_group": "full_image",
        "model_role": "architecture_matched_full_image_reference",
        "clinical_interpretation": "Reference full-image EfficientNet-B0 model used to isolate the effect of anatomical input restriction.",
    },
    "full_image__MobileNetV3Large": {
        "model_group": "full_image",
        "model_role": "full_image_finalist",
        "clinical_interpretation": "Lightweight full-image finalist from the classifier benchmark.",
    },
    "full_image__DeiTSmall": {
        "model_group": "full_image",
        "model_role": "full_image_finalist",
        "clinical_interpretation": "Transformer-based full-image finalist from the classifier benchmark.",
    },
    "anatomy__od_centered_crop_2p5x": {
        "model_group": "anatomy_constrained",
        "model_role": "primary_anatomically_parsimonious_model",
        "clinical_interpretation": "Primary OD-centered classifier; preserves local optic-disc/peripapillary context while restricting non-anatomical field information.",
    },
    "anatomy__od_centered_crop_4p0x": {
        "model_group": "anatomy_constrained",
        "model_role": "wider_peripapillary_context_sensitivity_model",
        "clinical_interpretation": "Sensitivity model with wider OD-centered context; validation-selected by Brier tie-break but less parsimonious than 2.5x.",
    },
    "anatomy__od_only_context_suppressed": {
        "model_group": "anatomy_constrained_stress_test",
        "model_role": "trained_od_only_context_suppressed_stress_test",
        "clinical_interpretation": "Stress-test classifier trained and evaluated with outside-OD context neutralized.",
    },
    "anatomy__outside_od_only_trained_control": {
        "model_group": "shortcut_control",
        "model_role": "trained_outside_od_shortcut_control",
        "clinical_interpretation": "Shortcut-control classifier; strong performance indicates predictive signal outside the optic disc and should not be treated as a clinically preferred model.",
    },
}

def get_metric_row(model_id, metric):
    block = primary_summary[
        (primary_summary["model_id"].astype(str) == model_id)
        & (primary_summary["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(f"Metric row not found uniquely: {model_id} | {metric}")

    return block.iloc[0]


def metric_value(model_id, metric, field):
    row = get_metric_row(model_id, metric)

    return float(row[field])


def get_primary_delta(comparator_model_id, metric):
    block = paired_comparison[
        (paired_comparison["reference_model_id"].astype(str) == PRIMARY_FULL_IMAGE_REFERENCE)
        & (paired_comparison["comparator_model_id"].astype(str) == comparator_model_id)
        & (paired_comparison["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(
            f"Primary delta row not found uniquely: {PRIMARY_FULL_IMAGE_REFERENCE} | {comparator_model_id} | {metric}"
        )

    return block.iloc[0]


model_role_rows = []

for model_id in EXPECTED_MODEL_IDS:
    role_info = MODEL_ROLES[model_id]

    model_role_rows.append(
        {
            "model_id": model_id,
            "model_group": role_info["model_group"],
            "model_role": role_info["model_role"],
            "clinical_interpretation": role_info["clinical_interpretation"],
            "patient_auroc": metric_value(model_id, "auroc", "observed"),
            "patient_auroc_ci95_low": metric_value(model_id, "auroc", "ci95_low"),
            "patient_auroc_ci95_high": metric_value(model_id, "auroc", "ci95_high"),
            "patient_balanced_accuracy": metric_value(model_id, "balanced_accuracy", "observed"),
            "patient_balanced_accuracy_ci95_low": metric_value(model_id, "balanced_accuracy", "ci95_low"),
            "patient_balanced_accuracy_ci95_high": metric_value(model_id, "balanced_accuracy", "ci95_high"),
            "patient_brier_score": metric_value(model_id, "brier_score", "observed"),
            "patient_brier_score_ci95_low": metric_value(model_id, "brier_score", "ci95_low"),
            "patient_brier_score_ci95_high": metric_value(model_id, "brier_score", "ci95_high"),
        }
    )

model_roles_df = pd.DataFrame(model_role_rows)
model_roles_df.to_csv(NOTEBOOK10_FINAL_MODEL_ROLES_CSV, index=False)

primary_delta_rows = []

for comparator_model_id in [
    PRIMARY_ANATOMICAL_MODEL,
    "anatomy__od_centered_crop_4p0x",
    OD_ONLY_STRESS_MODEL,
    OUTSIDE_OD_CONTROL_MODEL,
]:
    for metric in ["auroc", "balanced_accuracy", "brier_score"]:
        row = get_primary_delta(comparator_model_id, metric)

        primary_delta_rows.append(
            {
                "reference_model_id": PRIMARY_FULL_IMAGE_REFERENCE,
                "comparator_model_id": comparator_model_id,
                "metric": metric,
                "reference_observed": float(row["reference_observed"]),
                "comparator_observed": float(row["comparator_observed"]),
                "delta_comparator_minus_reference": float(row["delta_comparator_minus_reference"]),
                "delta_ci95_low": float(row["delta_ci95_low"]),
                "delta_ci95_high": float(row["delta_ci95_high"]),
                "paired_bootstrap_p_two_sided": float(row["paired_bootstrap_p_two_sided"]),
            }
        )

primary_delta_df = pd.DataFrame(primary_delta_rows)
primary_delta_df.to_csv(NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV, index=False)

interpretation_rows = [
    {
        "interpretation_item": "main_result",
        "interpretation": (
            "Anatomy-constrained classifiers preserved high patient-level discrimination compared with full-image classifiers. "
            "The OD-centered 2.5x model is the primary anatomically parsimonious candidate, while the OD-centered 4.0x model is a wider-context sensitivity comparator."
        ),
        "evidence_basis": (
            f"OD-centered 2.5x AUROC={metric_value(PRIMARY_ANATOMICAL_MODEL, 'auroc', 'observed'):.6f}; "
            f"OD-centered 4.0x AUROC={metric_value('anatomy__od_centered_crop_4p0x', 'auroc', 'observed'):.6f}."
        ),
    },
    {
        "interpretation_item": "shortcut_control_result",
        "interpretation": (
            "The outside-OD trained control achieved strong performance, indicating that HYGD contains predictive signal outside the optic disc. "
            "This supports the central methodological claim that high classification performance alone is insufficient evidence of anatomical plausibility."
        ),
        "evidence_basis": (
            f"Outside-OD trained control AUROC={metric_value(OUTSIDE_OD_CONTROL_MODEL, 'auroc', 'observed'):.6f}; "
            f"balanced_accuracy={metric_value(OUTSIDE_OD_CONTROL_MODEL, 'balanced_accuracy', 'observed'):.6f}; "
            f"Brier={metric_value(OUTSIDE_OD_CONTROL_MODEL, 'brier_score', 'observed'):.6f}."
        ),
    },
    {
        "interpretation_item": "stress_test_result",
        "interpretation": (
            "The OD-only context-suppressed stress-test model performed strongly when trained and evaluated under the same restricted-input distribution. "
            "This shows that OD-centered information remains highly predictive, but this condition should be interpreted as a stress test rather than a deployable clinical input mode."
        ),
        "evidence_basis": (
            f"OD-only trained stress-test AUROC={metric_value(OD_ONLY_STRESS_MODEL, 'auroc', 'observed'):.6f}; "
            f"balanced_accuracy={metric_value(OD_ONLY_STRESS_MODEL, 'balanced_accuracy', 'observed'):.6f}; "
            f"Brier={metric_value(OD_ONLY_STRESS_MODEL, 'brier_score', 'observed'):.6f}."
        ),
    },
    {
        "interpretation_item": "model_selection_policy",
        "interpretation": (
            "Although validation ranking selected OD-centered 4.0x by a Brier-score tie-break, the OD-centered 2.5x model remains the primary anatomically parsimonious comparator. "
            "The outside-OD model is retained as a shortcut-control, not as a clinical candidate."
        ),
        "evidence_basis": (
            f"Validation-selected experiment={validation_selected_payload['selected_experiment_id']}; "
            f"primary anatomically parsimonious model=od_centered_crop_2p5x; "
            f"outside-OD control=outside_od_only_trained_control."
        ),
    },
    {
        "interpretation_item": "next_step",
        "interpretation": (
            "The next notebook should train a hybrid anatomy-aware classifier combining a full-image branch with an OD-centered branch. "
            "The goal is to test whether global context and explicit anatomical focus can be combined while preserving performance and improving interpretability."
        ),
        "evidence_basis": "Notebook 10 completed controlled anatomy-constrained and shortcut-control experiments.",
    },
]

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_df.to_csv(NOTEBOOK10_FINAL_INTERPRETATION_CSV, index=False)

final_synthesis_rows = []

for _, row in model_roles_df.iterrows():
    final_synthesis_rows.append(
        {
            "model_id": row["model_id"],
            "model_group": row["model_group"],
            "model_role": row["model_role"],
            "patient_auroc": row["patient_auroc"],
            "patient_auroc_ci95": f"[{row['patient_auroc_ci95_low']:.6f}, {row['patient_auroc_ci95_high']:.6f}]",
            "patient_balanced_accuracy": row["patient_balanced_accuracy"],
            "patient_balanced_accuracy_ci95": f"[{row['patient_balanced_accuracy_ci95_low']:.6f}, {row['patient_balanced_accuracy_ci95_high']:.6f}]",
            "patient_brier_score": row["patient_brier_score"],
            "patient_brier_score_ci95": f"[{row['patient_brier_score_ci95_low']:.6f}, {row['patient_brier_score_ci95_high']:.6f}]",
            "clinical_interpretation": row["clinical_interpretation"],
        }
    )

final_synthesis_df = pd.DataFrame(final_synthesis_rows)
final_synthesis_df.to_csv(NOTEBOOK10_FINAL_SYNTHESIS_CSV, index=False)

validation_experiments = set(validation_ranking["experiment_id"].astype(str))
test_experiments = set(test_patient_metrics["experiment_id"].astype(str))
combined_model_ids = set(combined_predictions["model_id"].astype(str))
summary_model_ids = set(primary_summary["model_id"].astype(str))
paired_comparators = set(paired_comparison["comparator_model_id"].astype(str))

final_checks = pd.DataFrame(
    [
        {
            "check_name": "all_required_inputs_exist",
            "status": bool(input_audit["exists"].all()),
            "details": f"{int(input_audit['exists'].sum())}/{len(input_audit)} inputs found",
        },
        {
            "check_name": "all_anatomy_experiments_in_validation_ranking",
            "status": set(EXPECTED_ANATOMY_EXPERIMENTS).issubset(validation_experiments),
            "details": ",".join(sorted(validation_experiments)),
        },
        {
            "check_name": "all_anatomy_experiments_in_locked_test_metrics",
            "status": set(EXPECTED_ANATOMY_EXPERIMENTS).issubset(test_experiments),
            "details": ",".join(sorted(test_experiments)),
        },
        {
            "check_name": "all_expected_models_in_combined_predictions",
            "status": set(EXPECTED_MODEL_IDS).issubset(combined_model_ids),
            "details": ",".join(sorted(combined_model_ids)),
        },
        {
            "check_name": "all_expected_models_in_primary_summary",
            "status": set(EXPECTED_MODEL_IDS).issubset(summary_model_ids),
            "details": ",".join(sorted(summary_model_ids)),
        },
        {
            "check_name": "outside_od_control_in_paired_comparisons",
            "status": OUTSIDE_OD_CONTROL_MODEL in paired_comparators,
            "details": ",".join(sorted(paired_comparators)),
        },
        {
            "check_name": "figure_files_exist",
            "status": bool(figure_manifest["exists"].astype(bool).all()),
            "details": f"{int(figure_manifest['exists'].astype(bool).sum())}/{len(figure_manifest)} figures found",
        },
        {
            "check_name": "primary_anatomical_model_defined",
            "status": PRIMARY_ANATOMICAL_MODEL in summary_model_ids,
            "details": PRIMARY_ANATOMICAL_MODEL,
        },
        {
            "check_name": "next_notebook_defined",
            "status": True,
            "details": "11_hybrid_anatomy_aware_classifier_training.ipynb",
        },
    ]
)

final_checks.to_csv(NOTEBOOK10_FINAL_CHECKS_CSV, index=False)

if not final_checks["status"].astype(bool).all():
    print(final_checks.to_string(index=False))
    raise RuntimeError("One or more Notebook 10 final checks failed.")

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_safe(v) for v in value]

    if isinstance(value, tuple):
        return [json_safe(v) for v in value]

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        value = float(value)
        return None if not np.isfinite(value) else value

    if isinstance(value, float):
        return None if not np.isfinite(value) else value

    if pd.isna(value) if not isinstance(value, (list, dict, tuple)) else False:
        return None

    return value

final_status = {
    "notebook": "10_anatomy_constrained_classifier_training",
    "status": "PASS",
    "analysis_scope": "Anatomy-constrained and shortcut-control classifier training and comparison against full-image glaucoma classifiers.",
    "training_ready_strict_images": input_build_summary.get("n_training_ready_strict_images", None),
    "outside_od_control_images": outside_od_build_summary.get("n_manifest_rows", None),
    "models_included": EXPECTED_MODEL_IDS,
    "anatomy_experiments": EXPECTED_ANATOMY_EXPERIMENTS,
    "validation_selected_experiment": validation_selected_payload["selected_experiment_id"],
    "primary_anatomically_parsimonious_model": "od_centered_crop_2p5x",
    "outside_od_control_model": "outside_od_only_trained_control",
    "primary_full_image_reference": PRIMARY_FULL_IMAGE_REFERENCE,
    "primary_patient_level_results": final_synthesis_df.to_dict(orient="records"),
    "primary_interpretation": (
        "Notebook 10 shows that OD-centered anatomy-constrained classifiers preserve high patient-level discrimination, "
        "while the trained outside-OD shortcut-control also performs strongly. This supports the claim that high performance "
        "must be interpreted together with anatomical plausibility and shortcut-control analyses."
    ),
    "recommended_next_notebook": "11_hybrid_anatomy_aware_classifier_training.ipynb",
    "recommended_next_step": (
        "Train a hybrid anatomy-aware classifier combining a full-image branch with an OD-centered branch, then compare "
        "performance, calibration, XAI/anatomical plausibility, and shortcut robustness against the full-image and anatomy-constrained baselines."
    ),
    "outputs": {
        "final_synthesis_csv": str(NOTEBOOK10_FINAL_SYNTHESIS_CSV),
        "final_model_roles_csv": str(NOTEBOOK10_FINAL_MODEL_ROLES_CSV),
        "final_primary_deltas_csv": str(NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV),
        "final_interpretation_csv": str(NOTEBOOK10_FINAL_INTERPRETATION_CSV),
        "final_checks_csv": str(NOTEBOOK10_FINAL_CHECKS_CSV),
        "final_status_json": str(NOTEBOOK10_FINAL_STATUS_JSON),
        "final_status_txt": str(NOTEBOOK10_FINAL_STATUS_TXT),
        "main_comparison_figure_png": str(MAIN_COMPARISON_FIGURE_PNG),
        "main_comparison_figure_pdf": str(MAIN_COMPARISON_FIGURE_PDF),
        "main_comparison_figure_svg": str(MAIN_COMPARISON_FIGURE_SVG),
    },
}

with open(NOTEBOOK10_FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(final_status), f, indent=2)

status_lines = [
    "Notebook 10 final status: PASS",
    "",
    "Scope:",
    "Anatomy-constrained and shortcut-control classifier training and comparison against full-image glaucoma classifiers.",
    "",
    "Models included:",
]

for model_id in EXPECTED_MODEL_IDS:
    status_lines.append(f"- {model_id}")

status_lines.extend(
    [
        "",
        "Main interpretation:",
        final_status["primary_interpretation"],
        "",
        "Primary anatomy-constrained model:",
        "od_centered_crop_2p5x",
        "",
        "Validation-selected anatomy-constrained model:",
        str(validation_selected_payload["selected_experiment_id"]),
        "",
        "Shortcut-control model:",
        "outside_od_only_trained_control",
        "",
        "Recommended next notebook:",
        "11_hybrid_anatomy_aware_classifier_training.ipynb",
        "",
        f"Final synthesis CSV: {NOTEBOOK10_FINAL_SYNTHESIS_CSV}",
        f"Final model roles CSV: {NOTEBOOK10_FINAL_MODEL_ROLES_CSV}",
        f"Final primary deltas CSV: {NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV}",
        f"Final interpretation CSV: {NOTEBOOK10_FINAL_INTERPRETATION_CSV}",
        f"Final checks CSV: {NOTEBOOK10_FINAL_CHECKS_CSV}",
        f"Final status JSON: {NOTEBOOK10_FINAL_STATUS_JSON}",
    ]
)

NOTEBOOK10_FINAL_STATUS_TXT.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

required_outputs = [
    NOTEBOOK10_FINAL_SYNTHESIS_CSV,
    NOTEBOOK10_FINAL_MODEL_ROLES_CSV,
    NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV,
    NOTEBOOK10_FINAL_INTERPRETATION_CSV,
    NOTEBOOK10_FINAL_CHECKS_CSV,
    NOTEBOOK10_FINAL_STATUS_JSON,
    NOTEBOOK10_FINAL_STATUS_TXT,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Notebook 10 final outputs: {missing_outputs}")

print("Notebook 10 final synthesis:")
print(
    final_synthesis_df[
        [
            "model_id",
            "model_group",
            "model_role",
            "patient_auroc",
            "patient_balanced_accuracy",
            "patient_brier_score",
        ]
    ].to_string(index=False)
)

print("\nPrimary deltas vs full-image EfficientNetB0:")
print(primary_delta_df.to_string(index=False))

print("\nFinal interpretation:")
print(interpretation_df.to_string(index=False))

print("\nFinal checks:")
print(final_checks.to_string(index=False))

print(f"\nFinal synthesis CSV: {NOTEBOOK10_FINAL_SYNTHESIS_CSV}")
print(f"Final model roles CSV: {NOTEBOOK10_FINAL_MODEL_ROLES_CSV}")
print(f"Final primary deltas CSV: {NOTEBOOK10_FINAL_PRIMARY_DELTAS_CSV}")
print(f"Final interpretation CSV: {NOTEBOOK10_FINAL_INTERPRETATION_CSV}")
print(f"Final checks CSV: {NOTEBOOK10_FINAL_CHECKS_CSV}")
print(f"Final status JSON: {NOTEBOOK10_FINAL_STATUS_JSON}")
print(f"Final status TXT: {NOTEBOOK10_FINAL_STATUS_TXT}")

print("Status: PASS")